<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.fr/cap08/cap08.EPs_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

## 💻 **Partie Pratique avec Exercices de Programmation**

La présente liste d'exercices de programmation (EP) consolide les formulations théoriques présentées tout au long du Chapitre 8 — Correspondance de Caractéristiques, Détection d'Objets et Segmentation Classique — à travers un parcours pratique appliqué. Comme dans le chapitre précédent, les EP isolent les **grandeurs intermédiaires** de chaque technique — la distance entre descripteurs binaires, les termes d'une image intégrale, le comptage des *inliers* d'un modèle candidat, le chevauchement entre boîtes englobantes et l'étiquette de chaque composant connecté — permettant de valider manuellement chaque étape du raisonnement sans dépendre d'OpenCV ni d'images externes.

L'enchaînement des exercices reproduit le flux conceptuel du chapitre : on commence par la **distance de Hamming**, cœur de la correspondance des descripteurs binaires tels que ORB ; on avance vers le comptage des ***inliers*** qui soutient le **RANSAC** dans l'estimation robuste d'une homographie ; on poursuit avec l'**image intégrale**, l'astuce computationnelle qui rend le ***Haar Cascade*** viable en temps réel ; on approfondit l'**IoU et la Suppression Non-Maximale**, le post-traitement commun à pratiquement tout détecteur d'objets ; et on conclut avec l'**étiquetage des composants connectés**, l'approche classique — et ses limites — pour segmenter des instances individuelles dans un masque binaire.

### 🎯 Objectif de ce Carnet

Le carnet permet de développer, valider, organiser et tester des solutions d'**Exercices de Programmation (EPs)** dans des environnements interactifs, comme Colab, avec les mêmes cas de test que Moodle, en les y copiant uniquement au moment d'enregistrer la note officielle.

### *Téléchargement*

Téléchargez `morph.py` et `testsuite.py` en exécutant la cellule ci-dessous :

In [ ]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup(testsuite=True)
from morph import mm
from testsuite import TestSuite

#### Exécution des tests
Pour évaluer les tests, exécutez `TestSuite("EP08_01.extensão").run()` dans une nouvelle cellule, en remplaçant l’extension par celle du langage utilisé (`.py`, `.java`, `.c`, `.cpp`, `.js` ou `.r`). Le système télécharge les cas de test depuis GitHub, exécute le programme et calcule automatiquement la note.

Pour tester directement du code Python, sans enregistrer de fichier, utilisez `run_code(codigo)` en passant le code comme *chaîne de caractères* dans une variable `codigo` :

```python
codigo = """
# ... votre code ici ...
"""
TestSuite("EP08_01").run_code(codigo)
```

### 🛠️ Résumé des méthodes de `morph.py` (Chap. 8)

La bibliothèque `morph.py` met à disposition des fonctions pour l'analyse des composants connexes, l'extraction des contours, les métriques géométriques et les annotations :

1. **Composants et contours (`connectedComponents`, `findContours`)**
Étiquettent les régions connexes et extraient les contours des images binaires.
2. **Propriétés des contours (`contourArea`, `arcLength`, `convexHull`, `approxPolyDP`, `fitLine`)**
Calculent l'aire, le périmètre, l'enveloppe convexe, l'approximation polygonale et l'ajustement de droite pour un contour.
3. **Géométrie englobante (`boundingRect`, `minAreaRect`, `boxPoints`, `minEnclosingCircle`, `fitEllipse`)**
Déterminent les rectangles englobants (alignés ou orientés), les ellipses et le plus petit cercle circonscrit.
4. **Extraction et persistance des mesures (`measure`, `saveMeasures`)**
Extraient les descripteurs géométriques des objets (aire, circularité, solidité, centroïde) et exportent les données vers CSV, texte ou format YOLO.
5. **Évaluation et visualisation (`IoU`, `verifyBoundBox`, `showBoundBox`)**
Calculent l'intersection sur union (*Intersection over Union*), valident les boîtes englobantes avec des gabarits et dessinent les boîtes délimitantes annotées sur l'image.

### EP08_01 🟢 Distance de Hamming et correspondance de descripteurs binaires

L'ORB, utilisé dans le Projet Pratique 1 de ce chapitre, décrit le voisinage de chaque point d'intérêt comme une séquence de bits — et, par conséquent, la comparaison entre deux descripteurs n'utilise pas la distance euclidienne du k-NN du Chapitre 7, mais plutôt la **distance de Hamming** : le nombre de positions où les bits diffèrent. Avant d'appeler `cv2.BFMatcher(cv2.NORM_HAMMING)`, vous avez été chargé d'implémenter manuellement cette correspondance (*matching*) par force brute — la même étape qui, exécutée en interne par OpenCV, précède l'estimation robuste de l'homographie par RANSAC.

#### 📋 Directives d'implémentation

1. **Quantités :** Lire les entiers $N$ et $M$ — nombre de descripteurs extraits de l'image A et de l'image B, respectivement.
2. **Descripteurs de A :** Lire $N$ lignes, chacune contenant un descripteur binaire (une *chaîne* de caractères `0` et `1`, tous de même longueur).
3. **Descripteurs de B :** Lire $M$ lignes, dans le même format.
4. **Seuil :** Lire l'entier $\tau$ — distance de Hamming maximale acceptable pour considérer une correspondance valide.
5. **Distance de Hamming :** Pour deux descripteurs binaires $a$ et $b$ de même longueur,
$$
d_H(a, b) = \sum_{k} \mathbb{1}[a_k \neq b_k],
$$
   c'est-à-dire le nombre de positions où les bits diffèrent.
6. **Correspondance par voisin le plus proche :** Pour chaque descripteur $a_i$ de A ($i$ dans l'ordre de lecture, commençant à $0$), calculer sa distance de Hamming à **tous** les descripteurs de B et trouver celui de distance minimale. En cas d'égalité entre deux ou plusieurs descripteurs de B avec la même distance minimale, choisir celui de **plus petit indice**.
7. **Filtrage par seuil :** Si la distance minimale trouvée est $\le \tau$, la correspondance est valide ; sinon, $a_i$ n'a pas de correspondance.
8. **Sortie :** Pour chaque $i$ de $0$ à $N-1$, dans l'ordre de lecture, imprimer une ligne : `i j d` s'il existe une correspondance valide (où $j$ est l'indice du descripteur de B choisi et $d$ sa distance), ou `i -1` sinon. À la fin, imprimer `Total correspondances valides : X`.

#### 📌 Contraintes computationnelles

* **Même longueur :** tous les descripteurs (de A et de B) ont exactement le même nombre de bits.
* **Force brute :** comparer chaque descripteur de A à **tous** ceux de B — aucune indexation ni structure d'accélération n'est nécessaire.
* **Départage par plus petit indice en B**, et **jamais** par ordre de lecture de A (qui est déjà naturel, car chaque $a_i$ est traité de manière indépendante).

#### 🧠 Fondement théorique

| Élément | Rôle dans la correspondance ORB |
|---|---|
| Descripteur binaire (BRIEF) | Chaque bit est le résultat d'une comparaison d'intensité entre deux pixels du voisinage |
| Distance de Hamming | Métrique de dissimilarité entre *chaînes* binaires ; beaucoup plus rapide à calculer que la distance euclidienne (opération XOR + comptage de bits) |
| Voisin le plus proche | Critère de correspondance : chaque point de A est apparié au point de B avec le descripteur le plus similaire |
| Seuil $\tau$ | Filtre les correspondances peu fiables avant même le RANSAC — mais, comme discuté dans le chapitre, certaines correspondances incorrectes passent encore, exigeant la robustesse du RANSAC |

#### 📦 Spécification d'entrée et de sortie (VPL)

**Entrée :**

* Ligne 1 : Entiers $N$ et $M$.
* $N$ lignes suivantes : un descripteur binaire par ligne (*chaîne* de `0` et de `1`).
* $M$ lignes suivantes : un descripteur binaire par ligne, dans le même format.
* Dernière ligne : Entier $\tau$.

**Sortie :**

* $N$ lignes, une par descripteur de A, au format `i j d` ou `i -1`.
* Dernière ligne : `Total correspondances valides : X`.

#### 📌 Exemples

| Entrée | Sortie | Observation |
|---|---|---|
| 3 3<br>10101010<br>11110000<br>00001111<br>10101011<br>00001110<br>11111111<br>2 | 0 0 1<br>1 -1<br>2 1 1<br>Total correspondances valides : 2 | Le descripteur `11110000` ne trouve pas de correspondance : son voisin le plus proche est à distance 4, au-dessus du seuil $\tau=2$. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0801" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0801 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0801 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0801 button:hover { background: #e8dfcf; }
  .sim-ep0801_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0801_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0801_bit { width: 36px; height: 36px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-family: monospace; font-weight: 700; font-size: 14px; user-select: none; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulateur EP08_01 : Distance de Hamming entre descripteurs binaires</span>
  <span class="sim-ep0801_pill">Descripteurs de 8 bits</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel Informativo de Instruções -->
  <div class="sim-ep0801_panel" style="margin-bottom:14px;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; text-align:center;">
      Cliquez sur n'importe quel bit du <b>Descripteur B</b> pour l'inverser et observez la distance de Hamming changer en temps réel.
    </div>
  </div>

  <!-- Grades dos Descritores -->
  <div class="sim-ep0801_panel" style="margin-bottom:14px; display:flex; flex-direction:column; gap:12px; align-items:center;">
    <div>
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:6px; text-align:center; letter-spacing:0.04em;">
        Descripteur A (Fixe)
      </div>
      <div id="sim-ep0801_a" style="display:grid; grid-template-columns:repeat(8, 36px); gap:4px;"></div>
    </div>

    <div>
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:6px; text-align:center; letter-spacing:0.04em;">
        Descripteur B (Cliquer pour inverser)
      </div>
      <div id="sim-ep0801_b" style="display:grid; grid-template-columns:repeat(8, 36px); gap:4px;"></div>
    </div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0801_debug" class="sim-ep0801_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim08Ep01(root){
    if (!root || root.dataset.sim08Ep01Init) return;
    root.dataset.sim08Ep01Init = "1";

    var A = [1, 0, 1, 0, 1, 0, 1, 0];
    var B = [1, 0, 1, 0, 1, 0, 1, 1];

    var aEl = root.querySelector('#sim-ep0801_a');
    var bEl = root.querySelector('#sim-ep0801_b');
    var dbg = root.querySelector('#sim-ep0801_debug');

    function estiloBit(destacado, interativo){
      var base = destacado 
        ? 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;' 
        : 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;';
      var cursor = interativo ? ' cursor:pointer;' : ' cursor:default;';
      return base + cursor;
    }

    function render(){
      aEl.innerHTML = ''; 
      bEl.innerHTML = '';
      var dist = 0;

      for (var k = 0; k < 8; k++){
        var diff = A[k] !== B[k];
        if (diff) dist++;

        var da = document.createElement('div');
        da.className = 'sim-ep0801_bit';
        da.style.cssText = estiloBit(diff, false);
        da.textContent = A[k];
        aEl.appendChild(da);

        var db = document.createElement('div');
        db.className = 'sim-ep0801_bit';
        db.style.cssText = estiloBit(diff, true);
        db.textContent = B[k];
        
        (function(idx){
          db.addEventListener('click', function(){
            B[idx] = 1 - B[idx];
            render();
          });
        })(k);

        bEl.appendChild(db);
      }

      if (dist === 0) {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        dbg.style.borderColor = '#e4dcc8';
        dbg.style.background  = '#fafaf7';
        dbg.style.color       = '#26241d';
      }

      dbg.textContent = 'A = ' + A.join('') + '   B = ' + B.join('') + '   →   Distância de Hamming = ' + dist;
    }

    render();
  }

  function tryInitSim08Ep01(){
    var root = document.getElementById('sim-ep0801');
    if (root) initSim08Ep01(root); else setTimeout(tryInitSim08Ep01, 200);
  }
  tryInitSim08Ep01();
})();
</script>
""")

**Figure 8.1:** Simulateur EP08_01 : Distance de Hamming entre deux descripteurs binaires


In [ ]:
%%writefile EP08_01.py
# Code Python

In [ ]:
TestSuite("EP08_01.py").run()

### EP08_02 🟢 Homographie et RANSAC : Le vote par *inliers*

Le RANSAC, présenté dans la section « Modélisation mathématique : Homographie et RANSAC », répète un cycle de trois étapes — tirer un échantillon minimal, estimer un modèle candidat, et compter combien de correspondances sont cohérentes avec lui (les ***inliers***) — en conservant à la fin le modèle le plus voté. L'étape d'estimation du modèle à partir de 4 points (étape 2) implique une algèbre linéaire qui sort du cadre de cet EP ; ici, vous recevez directement un ensemble d'homographies **déjà candidates** — comme si chacune avait été estimée à partir d'un échantillon aléatoire différent — et vous êtes chargé de reproduire exactement l'étape décisive de l'algorithme : **appliquer chaque modèle à toutes les correspondances et compter ses *inliers***, en choisissant le gagnant.

#### 📋 Directives d'implémentation

1. **Correspondances :** Lire l'entier $N$ puis $N$ lignes contenant chacune quatre réels, $x\ y\ x'\ y'$ — un point de l'image A et son correspondant (éventuellement incorrect) dans l'image B, exactement comme produit par l'étape de *matching* de l'EP08_01.
2. **Modèles candidats :** Lire l'entier $K$ (nombre d'homographies candidates) et le réel $\varepsilon$ (seuil d'erreur de reprojection). Ensuite, lire $K$ lignes, chacune avec neuf réels $h_{11}\ h_{12}\ h_{13}\ h_{21}\ h_{22}\ h_{23}\ h_{31}\ h_{32}\ h_{33}$ — les éléments de la matrice $H$ candidate, en ordre de lecture par ligne (*row-major*).
3. **Reprojection :** Pour chaque correspondance $(x,y,x',y')$ et chaque modèle candidat $H_k$, calculer le point projeté
$$
\begin{bmatrix} \hat x \\ \hat y \\ \hat w \end{bmatrix} = H_k \begin{bmatrix} x \\ y \\ 1 \end{bmatrix},
\qquad
(\hat x / \hat w,\ \hat y / \hat w)\ \text{est le point projeté.}
$$
4. **Erreur de reprojection :** $e = \sqrt{(\hat x/\hat w - x')^2 + (\hat y /\hat w - y')^2}$.
5. **Comptage des *inliers* :** Une correspondance est un *inlier* du modèle $H_k$ si $e \le \varepsilon$.
6. **Sélection du meilleur modèle :** Le modèle gagnant est celui ayant le plus grand nombre d'*inliers* ; en cas d'égalité, choisissez celui de **plus petit indice** $k$ (le premier trouvé pendant le cycle itératif du RANSAC).
7. **Sortie :** Pour chaque modèle $k$ de $0$ à $K-1$, dans l'ordre de lecture, imprimer `Modelo k: I inliers`. À la fin, imprimer `Melhor modelo: k_best com I_best inliers`.

#### 📌 Contraintes computationnelles

* **Comparaison inclusive :** une erreur de reprojection **exactement égale** à $\varepsilon$ compte comme *inlier* ($e \le \varepsilon$).
* **Sans estimation de $H$ :** les matrices sont déjà fournies prêtes — il n'est pas nécessaire (ni attendu) de résoudre un quelconque système linéaire.
* **Égalité résolue par le plus petit indice**, reflétant le comportement naturel d'un algorithme itératif qui parcourt les modèles dans l'ordre et ne remplace le meilleur trouvé jusqu'alors que lorsqu'un nouveau modèle le **dépasse strictement**.

#### 🧠 Fondement théorique

| Élément | Rôle dans le RANSAC |
|---|---|
| Échantillon minimal (4 paires) | Suffisant pour déterminer les 8 degrés de liberté d'une homographie |
| Modèle candidat $H_k$ | Estimé à partir d'un échantillon minimal ; peut être bon ou mauvais, selon que l'échantillon contenait des *outliers* |
| Erreur de reprojection | Mesure à quel point le modèle « prédit » chaque correspondance observée |
| *Inlier* vs. *outlier* | Correspondances cohérentes avec le modèle gagnant (*inliers*) vs. les autres, typiquement des correspondances incorrectes du *matching* |
| Raffinement final | En pratique, après avoir choisi le meilleur modèle, le RANSAC le recalcule en utilisant **uniquement** ses *inliers* — étape non exigée dans cet EP |

#### 📦 Spécification d'entrée et de sortie (VPL)

**Entrée :**

* Ligne 1 : Entier $N$.
* $N$ lignes suivantes : quatre réels $x\ y\ x'\ y'$.
* Ligne suivante : Entier $K$ et réel $\varepsilon$.
* $K$ lignes suivantes : neuf réels (éléments de $H_k$, *row-major*).

**Sortie :**

* $K$ lignes au format `Modelo k: I inliers`.
* Dernière ligne : `Melhor modelo: k_best com I_best inliers`.

#### 📌 Exemples

| Entrée | Sortie | Observation |
|---|---|---|
| 5<br>0 0 0 0<br>1 1 2 2<br>2 0 4 0<br>0 2 0 4<br>5 5 1 1<br>2 0.5<br>2 0 0 0 2 0 0 0 1<br>1 0 0 0 1 0 0 0 1 | Modelo 0: 4 inliers<br>Modelo 1: 1 inliers<br>Melhor modelo: 0 com 4 inliers | Le Modèle 0 (échelle ×2) explique correctement 4 des 5 correspondances ; la 5ᵉ, $(5,5)\to(1,1)$, est un *outlier* qu'aucun des deux modèles n'explique bien. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0802" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0802 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0802 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0802 button:hover { background: #e8dfcf; }
  #sim-ep0802 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0802_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0802_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulateur EP08_02 : RANSAC &mdash; Comptage des inliers</span>
  <span class="sim-ep0802_pill">Modèle : Échelle &times;2</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0808_panel sim-ep0802_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Seuil (&epsilon;): <span id="sim-ep0802_vl" style="font-family:monospace; color:#26241d;">0.50</span>
      </label>
    </div>
    
    <input id="sim-ep0802_sl" type="range" min="0" max="13" step="0.25" value="0.5">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Le modèle candidat mappe (x,y) &rarr; à (2x,2y). Ajustez le seuil &epsilon; et observez quelles correspondances deviennent des inliers ou des outliers.
    </div>
  </div>

  <!-- Cards de Pontos / Correspondências -->
  <div id="sim-ep0802_cards" style="display:grid; grid-template-columns: repeat(auto-fit, minmax(110px, 1fr)); gap:8px; margin-bottom:14px;"></div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0802_debug" class="sim-ep0802_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim08Ep02(root){
    if (!root || root.dataset.sim08Ep02Init) return;
    root.dataset.sim08Ep02Init = "1";

    var pontos = [
      {x:0, y:0, xp:0, yp:0},
      {x:1, y:1, xp:2, yp:2},
      {x:2, y:0, xp:4, yp:0},
      {x:0, y:2, xp:0, yp:4},
      {x:5, y:5, xp:1, yp:1}
    ];

    var slEl  = root.querySelector('#sim-ep0802_sl');
    var vlEl  = root.querySelector('#sim-ep0802_vl');
    var cards = root.querySelector('#sim-ep0802_cards');
    var dbg   = root.querySelector('#sim-ep0802_debug');

    function render(){
      var eps = parseFloat(slEl.value);
      vlEl.textContent = eps.toFixed(2);
      cards.innerHTML = '';
      var inliers = 0;

      pontos.forEach(function(p){
        var px = 2 * p.x, py = 2 * p.y;
        var erro = Math.sqrt((px - p.xp) * (px - p.xp) + (py - p.yp) * (py - p.yp));
        var dentro = erro <= eps;
        if (dentro) inliers++;

        var div = document.createElement('div');
        div.style.cssText = 'text-align:center; border-radius:10px; padding:10px 6px; font-size:11px; transition:all 0.15s ease;' +
          (dentro 
            ? 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;' 
            : 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;');

        div.innerHTML = '<div style="font-weight:700; margin-bottom:4px;">(' + p.x + ',' + p.y + ') &rarr; (' + p.xp + ',' + p.yp + ')</div>' +
          '<div style="font-family:monospace; margin-bottom:4px; font-size:10px;">erro = ' + erro.toFixed(2) + '</div>' +
          '<div style="font-weight:700; font-size:10px;">' + (dentro ? 'INLIER' : 'outlier') + '</div>';

        cards.appendChild(div);
      });

      if (inliers > 0) {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      }

      dbg.textContent = '\u03B5 = ' + eps.toFixed(2) + '  |  inliers = ' + inliers + ' de ' + pontos.length;
    }

    slEl.addEventListener('input', render);
    render();
  }

  function tryInitSim08Ep02(){
    var root = document.getElementById('sim-ep0802');
    if (root) initSim08Ep02(root); else setTimeout(tryInitSim08Ep02, 200);
  }
  tryInitSim08Ep02();
})();
</script>
""")

**Figure 8.2:** Simulateur EP08_02: RANSAC — Vote par Inliers entre Modèles Candidats


In [ ]:
%%writefile EP08_02.py
# Code Python

In [ ]:
TestSuite("EP08_02.py").run()

### EP08_03 🟢 Image intégrale : sommes rectangulaires en temps constant

Imaginez une caméra de sécurité traitant 30 images par seconde, et pour chaque image, le système doit balayer l'image à des dizaines de positions et d'échelles différentes, testant à chaque fois un ensemble de caractéristiques rectangulaires pour décider « y a-t-il un visage ici ? ». Si le calcul de la somme des intensités de chaque rectangle exigeait de sommer pixel par pixel, le système n'aurait aucune chance de fonctionner en temps réel — le goulot d'étranglement se situerait précisément dans la partie la plus répétée de l'algorithme. C'est exactement ce goulot d'étranglement que l'image intégrale élimine.

Le détecteur Haar Cascade évalue des milliers de caractéristiques rectangulaires par fenêtre, à de multiples positions et échelles — une approche irréalisable en temps réel si chaque rectangle exigeait la somme de ses pixels un par un. L'**image intégrale**, définie dans la section consacrée au Haar Cascade, résout ce problème : une fois précalculée, la somme des intensités de **n'importe quelle** région rectangulaire s'obtient avec seulement quatre consultations et trois opérations arithmétiques, quelle que soit la taille du rectangle.

Vous êtes chargé d'implémenter cette structure à partir de zéro : d'abord, calculer l'image intégrale à partir de l'image originale ; ensuite, répondre aux requêtes rectangulaires arbitraires.

#### 📋 Directives d'implémentation

1. **Entrée :** Lire les dimensions $H \times W$ de l'image et ses $H \times W$ valeurs entières d'intensité.
2. **Image intégrale :** Calculer, pour chaque position $(i,j)$ (indexation à partir de $0$, `[ligne][colonne]`),
$$
II(i,j) = \sum_{i' \le i,\ j' \le j} I(i', j'),
$$
   c'est-à-dire la somme de tous les pixels au-dessus et à gauche de $(i,j)$, y compris la position elle-même.
3. **Requêtes :** Lire l'entier $Q$, puis $Q$ lignes, chacune avec quatre entiers $x_1\ y_1\ x_2\ y_2$ — les coins supérieur-gauche et inférieur-droit d'un rectangle, **tous deux inclusifs**, avec $0 \le x_1 \le x_2 < W$ et $0 \le y_1 \le y_2 < H$.
4. **Somme rectangulaire en O(1) :** Pour chaque requête, calculer la somme des intensités dans le rectangle en utilisant uniquement des valeurs déjà présentes dans $II$ (sans parcourir les pixels originaux) :
$$
S(x_1,y_1,x_2,y_2) = II(y_2,x_2) - II(y_2, x_1{-}1) - II(y_1{-}1, x_2) + II(y_1{-}1, x_1{-}1),
$$
   en traitant tout terme dont l'indice de ligne ou de colonne est égal à $-1$ comme $0$.
5. **Sortie :** D'abord, imprimer l'image intégrale complète — $H$ lignes avec $W$ entiers chacune. Ensuite, pour chaque requête, imprimer un seul entier : la somme de la région correspondante.

#### 📌 Contraintes de calcul

* **Ne recalculez pas par force brute :** la réponse à chaque requête doit utiliser la formule à quatre termes sur $II$, et non une somme directe des pixels du rectangle (même si le résultat numérique est le même, l'objectif de l'exercice est précisément cette technique).
* **Rectangles à coordonnées inclusives :** $(x_1,y_1)$ et $(x_2,y_2)$ appartiennent à la région sommée.
* **Traitement des bords :** lors de la consultation de $II$ avec l'indice $-1$ (lorsque $x_1=0$ ou $y_1=0$), utiliser la valeur $0$.

#### 🧠 Fondement théorique

| Élément | Rôle dans le détecteur Haar Cascade |
|---|---|
| Image intégrale $II$ | Précalculée une seule fois par image, en temps $O(HW)$ |
| Requête en O(1) | Chaque caractéristique Haar (différence entre les sommes de régions rectangulaires) est évaluée avec peu d'opérations, quelle que soit l'aire du rectangle |
| Scalabilité | C'est cette constance qui rend possible l'évaluation de milliers de caractéristiques, à de multiples positions et échelles, en temps réel |
| Principe d'inclusion-exclusion | Les quatre termes de la formule additionnent la région souhaitée et soustraient exactement les zones comptées en double |

#### 📦 Spécification des entrées et sorties (VPL)

**Entrée :**

* Ligne 1 : Entiers $H$ et $W$.
* $H$ lignes suivantes : $W$ entiers chacune (image originale).
* Ligne suivante : Entier $Q$.
* $Q$ lignes suivantes : quatre entiers $x_1\ y_1\ x_2\ y_2$.

**Sortie :**

* $H$ lignes avec $W$ entiers chacune (image intégrale).
* $Q$ lignes, une par requête, avec la somme de la région correspondante.

#### 📌 Exemples

| Entrée | Sortie | Observation |
|---|---|---|
| 3 3<br>1 2 3<br>4 5 6<br>7 8 9<br>1<br>0 0 2 2 | 1 3 6<br>5 12 21<br>12 27 45<br>45 | La requête couvre l'image entière ; la somme coïncide avec $II(2,2)$ et avec la somme des 9 valeurs. |
| 3 3<br>1 2 3<br>4 5 6<br>7 8 9<br>2<br>1 1 2 2<br>0 0 1 1 | 1 3 6<br>5 12 21<br>12 27 45<br>28<br>12 | La première requête utilise les quatre termes de la formule ; la seconde coïncide directement avec $II(1,1)$, car elle commence à l'origine. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0803" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0803 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0803 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0803 button:hover { background: #e8dfcf; }
  #sim-ep0803 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0803_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0803_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulateur EP08_03 : Somme rectangulaire avec image intégrale</span>
  <span id="sim-ep0803_badge" class="sim-ep0803_pill">Interne</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Descrição e Botões de Preset -->
  <div class="sim-ep0803_panel" style="margin-bottom:14px;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; text-align:center; margin-bottom:10px;">
      Choisissez un rectangle (x<sub>1</sub>, y<sub>1</sub>) &ndash; (x<sub>2</sub>, y<sub>2</sub>). L'image intégrale II inclut une bordure virtuelle (&minus;1) avec des zéros pour une validation sans exceptions.
    </div>

    <div style="display:flex; gap:6px; justify-content:center; flex-wrap:wrap;">
      <button data-preset="0,0,3,3">Depuis l'origine</button>
      <button data-preset="0,1,2,3">Bordure gauche</button>
      <button data-preset="1,0,3,2">Bordure supérieure</button>
      <button data-preset="1,1,2,2">Totalement interne</button>
      <button data-preset="2,2,2,2">Pixel unique</button>
    </div>
  </div>

  <!-- Sliders de Seleção das Coordenadas -->
  <div class="sim-ep0803_panel" style="margin-bottom:14px;">
    <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:12px;">
      
      <div>
        <div style="font-size:11px; font-weight:700; color:#5e5a4a; margin-bottom:4px;">
          Coin supérieur-gauche (x<sub>1</sub>, y<sub>1</sub>) = <span id="sim-ep0803_v_tl" style="font-family:monospace; color:#26241d;">(1,1)</span>
        </div>
        <div style="display:flex; justify-content:space-between; align-items:center; font-size:10px; color:#8a8371; font-weight:600;">x<sub>1</sub></div>
        <input id="sim-ep0803_x1" type="range" min="0" max="3" step="1" value="1">
        
        <div style="display:flex; justify-content:space-between; align-items:center; font-size:10px; color:#8a8371; font-weight:600; margin-top:4px;">y<sub>1</sub></div>
        <input id="sim-ep0803_y1" type="range" min="0" max="3" step="1" value="1">
      </div>

      <div>
        <div style="font-size:11px; font-weight:700; color:#5e5a4a; margin-bottom:4px;">
          Coin inférieur-droit (x<sub>2</sub>, y<sub>2</sub>) = <span id="sim-ep0803_v_br" style="font-family:monospace; color:#26241d;">(2,2)</span>
        </div>
        <div style="display:flex; justify-content:space-between; align-items:center; font-size:10px; color:#8a8371; font-weight:600;">x<sub>2</sub></div>
        <input id="sim-ep0803_x2" type="range" min="0" max="3" step="1" value="2">
        
        <div style="display:flex; justify-content:space-between; align-items:center; font-size:10px; color:#8a8371; font-weight:600; margin-top:4px;">y<sub>2</sub></div>
        <input id="sim-ep0803_y2" type="range" min="0" max="3" step="1" value="2">
      </div>

    </div>
  </div>

  <!-- Grades das Matrizes -->
  <div style="display:flex; gap:20px; justify-content:center; flex-wrap:wrap; margin-bottom:14px;">
    <div class="sim-ep0803_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:8px; letter-spacing:0.04em;">
        Image originale I (4&times;4)
      </div>
      <div id="sim-ep0803_gridI" style="display:grid; justify-content:center;"></div>
    </div>

    <div class="sim-ep0803_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:8px; letter-spacing:0.04em;">
        Image intégrale II (Avec bordure virtuelle &minus;1)
      </div>
      <div id="sim-ep0803_gridII" style="display:grid; justify-content:center;"></div>
    </div>
  </div>

  <!-- Legenda das Operações -->
  <div style="display:flex; gap:12px; justify-content:center; flex-wrap:wrap; margin-bottom:14px; font-size:10px; font-weight:700; color:#5e5a4a;">
    <span style="display:flex; align-items:center; gap:4px;"><span style="width:10px; height:10px; background:#2980b9; border-radius:2px; display:inline-block;"></span> + II(y<sub>2</sub>, x<sub>2</sub>)</span>
    <span style="display:flex; align-items:center; gap:4px;"><span style="width:10px; height:10px; background:#d35400; border-radius:2px; display:inline-block;"></span> &minus; II(y<sub>2</sub>, x<sub>1</sub>&minus;1)</span>
    <span style="display:flex; align-items:center; gap:4px;"><span style="width:10px; height:10px; background:#d35400; border-radius:2px; display:inline-block;"></span> &minus; II(y<sub>1</sub>&minus;1, x<sub>2</sub>)</span>
    <span style="display:flex; align-items:center; gap:4px;"><span style="width:10px; height:10px; background:#27ae60; border-radius:2px; display:inline-block;"></span> + II(y<sub>1</sub>&minus;1, x<sub>1</sub>&minus;1)</span>
  </div>

  <!-- Painéis Informativos / Resultados -->
  <div id="sim-ep0803_formula" class="sim-ep0803_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center; margin-bottom:8px;"></div>
  <div id="sim-ep0803_verify" class="sim-ep0803_panel" style="font-family:monospace; font-size:11px; color:#04342C; background:#eafaf1; border-color:#a3e4d7; text-align:center;"></div>

</div>
</div>

<script>
(function(){
  function initSim08Ep03(root){
    if (!root || root.dataset.sim08Ep03Init) return;
    root.dataset.sim08Ep03Init = "1";

    var I = [
      [2, 1, 3, 4],
      [5, 6, 1, 2],
      [3, 2, 4, 1],
      [1, 3, 2, 5]
    ];
    var N = 4;
    var II = [];
    for (var i = 0; i < N; i++){ II.push([0, 0, 0, 0]); }
    
    for (var i = 0; i < N; i++){
      for (var j = 0; j < N; j++){
        II[i][j] = I[i][j] +
          (i > 0 ? II[i - 1][j] : 0) + 
          (j > 0 ? II[i][j - 1] : 0) - 
          (i > 0 && j > 0 ? II[i - 1][j - 1] : 0);
      }
    }

    var x1El     = root.querySelector('#sim-ep0803_x1');
    var y1El     = root.querySelector('#sim-ep0803_y1');
    var x2El     = root.querySelector('#sim-ep0803_x2');
    var y2El     = root.querySelector('#sim-ep0803_y2');
    var vTl      = root.querySelector('#sim-ep0803_v_tl');
    var vBr      = root.querySelector('#sim-ep0803_v_br');
    var gridI    = root.querySelector('#sim-ep0803_gridI');
    var gridII   = root.querySelector('#sim-ep0803_gridII');
    var formulaEl= root.querySelector('#sim-ep0803_formula');
    var verifyEl = root.querySelector('#sim-ep0803_verify');
    var badge    = root.querySelector('#sim-ep0803_badge');

    var CELL = 34, HEAD = 20;

    function cellDiv(text, size, extraStyle){
      var d = document.createElement('div');
      d.style.cssText = 'display:flex; align-items:center; justify-content:center; font-family:monospace; font-size:' + size + 'px;' + extraStyle;
      d.textContent = text;
      return d;
    }

    function clampAndRender(changed){
      var x1 = +x1El.value, y1 = +y1El.value, x2 = +x2El.value, y2 = +y2El.value;
      if (changed === 'x1' && x1 > x2) x2El.value = x1;
      if (changed === 'x2' && x2 < x1) x1El.value = x2;
      if (changed === 'y1' && y1 > y2) y2El.value = y1;
      if (changed === 'y2' && y2 < y1) y1El.value = y2;
      render();
    }

    function render(){
      var x1 = +x1El.value, y1 = +y1El.value, x2 = +x2El.value, y2 = +y2El.value;
      vTl.textContent = '(' + x1 + ',' + y1 + ')';
      vBr.textContent = '(' + x2 + ',' + y2 + ')';

      var sit;
      if (x1 === x2 && y1 === y2) sit = 'Pixel Único';
      else if (x1 === 0 && y1 === 0) sit = 'Desde a Origem';
      else if (x1 === 0) sit = 'Borda Esquerda';
      else if (y1 === 0) sit = 'Borda Superior';
      else sit = 'Interno';

      badge.textContent = sit;

      // Grade I
      gridI.style.gridTemplateColumns = HEAD + 'px repeat(' + N + ',' + CELL + 'px)';
      gridI.style.gridTemplateRows = HEAD + 'px repeat(' + N + ',' + CELL + 'px)';
      gridI.innerHTML = '';
      gridI.appendChild(cellDiv('', 10, 'color:#8a8371;'));
      for (var c = 0; c < N; c++) gridI.appendChild(cellDiv(c, 10, 'color:#8a8371; font-weight:700;'));
      
      for (var r = 0; r < N; r++){
        gridI.appendChild(cellDiv(r, 10, 'color:#8a8371; font-weight:700;'));
        for (var c = 0; c < N; c++){
          var dentro = (r >= y1 && r <= y2 && c >= x1 && c <= x2);
          gridI.appendChild(cellDiv(I[r][c], 12, 'border-radius:4px; transition:all 0.15s ease;' +
            (dentro 
              ? 'background:#f1ead7; border:2px solid #26241d; font-weight:700; color:#26241d;'
              : 'background:#fafaf7; border:1px solid #e4dcc8; color:#8a8371;')));
        }
      }

      // Grade II (5x5 dados)
      var M = N + 1;
      gridII.style.gridTemplateColumns = HEAD + 'px repeat(' + M + ',' + CELL + 'px)';
      gridII.style.gridTemplateRows = HEAD + 'px repeat(' + M + ',' + CELL + 'px)';
      gridII.innerHTML = '';
      gridII.appendChild(cellDiv('', 10, 'color:#8a8371;'));
      for (var c2 = 0; c2 < M; c2++) gridII.appendChild(cellDiv(c2 - 1, 10, 'color:#8a8371; font-weight:700;'));

      var t1 = [y2 + 1, x2 + 1];
      var t2 = [y2 + 1, x1];
      var t3 = [y1, x2 + 1];
      var t4 = [y1, x1];

      function styleFor(r2, c2){
        var isVirtual = (r2 === 0 || c2 === 0);
        var base = isVirtual
          ? 'border-radius:4px; background:#fafaf7; border:1px dashed #e4dcc8; color:#8a8371;'
          : 'border-radius:4px; background:#fafaf7; border:1px solid #e4dcc8; color:#26241d;';
        
        function match(t, color, tcolor){
          if (r2 === t[0] && c2 === t[1]) {
            return 'border-radius:4px; font-weight:700; background:' + color + '; border:2px solid ' + tcolor + '; color:#ffffff;';
          }
          return null;
        }

        return match(t1, '#2980b9', '#1c5d85') || 
               match(t2, '#d35400', '#a04000') ||
               match(t3, '#d35400', '#a04000') || 
               match(t4, '#27ae60', '#1e8449') || base;
      }

      for (var r2 = 0; r2 < M; r2++){
        gridII.appendChild(cellDiv(r2 - 1, 10, 'color:#8a8371; font-weight:700;'));
        for (var c2 = 0; c2 < M; c2++){
          var val = (r2 === 0 || c2 === 0) ? 0 : II[r2 - 1][c2 - 1];
          gridII.appendChild(cellDiv(val, 12, styleFor(r2, c2)));
        }
      }

      function term(y, x){ return (y < 0 || x < 0) ? 0 : II[y][x]; }
      var a = term(y2, x2), b = term(y2, x1 - 1), c3 = term(y1 - 1, x2), d = term(y1 - 1, x1 - 1);
      var S = a - b - c3 + d;

      formulaEl.innerHTML =
        'S = II(' + y2 + ',' + x2 + ') &minus; II(' + y2 + ',' + (x1 - 1) + ') &minus; II(' + (y1 - 1) + ',' + x2 + ') + II(' + (y1 - 1) + ',' + (x1 - 1) + ')<br>' +
        'S = ' + a + ' &minus; ' + b + ' &minus; ' + c3 + ' + ' + d + ' = <b>' + S + '</b>';

      var direta = 0;
      for (var rr = y1; rr <= y2; rr++){
        for (var cc = x1; cc <= x2; cc++){
          direta += I[rr][cc];
        }
      }

      if (direta === S) {
        verifyEl.style.borderColor = '#a3e4d7';
        verifyEl.style.background  = '#eafaf1';
        verifyEl.style.color       = '#04342C';
      } else {
        verifyEl.style.borderColor = '#f5b7b1';
        verifyEl.style.background  = '#fdecea';
        verifyEl.style.color       = '#c0392b';
      }

      verifyEl.innerHTML = '&#10004; Verificação (Soma Direta dos Pixels) = ' + direta + (direta === S ? ' &rarr; Bate com S' : ' &rarr; Erro');
    }

    x1El.addEventListener('input', function(){ clampAndRender('x1'); });
    y1El.addEventListener('input', function(){ clampAndRender('y1'); });
    x2El.addEventListener('input', function(){ clampAndRender('x2'); });
    y2El.addEventListener('input', function(){ clampAndRender('y2'); });

    root.querySelectorAll('button[data-preset]').forEach(function(btn){
      btn.addEventListener('click', function(){
        var p = btn.getAttribute('data-preset').split(',').map(Number);
        x1El.value = p[0]; y1El.value = p[1]; x2El.value = p[2]; y2El.value = p[3];
        render();
      });
    });

    render();
  }

  function tryInitSim08Ep03(){
    var root = document.getElementById('sim-ep0803');
    if (root) initSim08Ep03(root); else setTimeout(tryInitSim08Ep03, 200);
  }
  tryInitSim08Ep03();
})();
</script>
""")

**Figure 8.3:** Simulateur EP08_03 : Somme Rectangulaire en O(1) — Multiples Situations de Bord


In [ ]:
%%writefile EP08_03.py
# Code Python

In [ ]:
TestSuite("EP08_03.py").run()

### EP08_04 🟢 IoU et Suppression des Non-Maximums (NMS)

La figure de cette section a montré l’effet de la Suppression des Non-Maximums sur un ensemble de boîtes produites par un détecteur de type *sliding window* : de multiples détections redondantes par objet ont été réduites à une seule boîte par objet. Vous avez été chargé de réimplémenter, octet par octet, les deux fonctions qui ont produit ce résultat — `calcular_iou` et `supressao_nao_maximos` — afin de confirmer, de vos propres mains, exactement les nombres que le chapitre a présentés.

#### 📋 Directives d’implémentation

1. **Entrée :** Lire l’entier $N$ (nombre de boîtes) et le réel $\tau$ (seuil d’IoU). Ensuite, lire $N$ lignes, chacune avec cinq réels $x_{min}\ y_{min}\ x_{max}\ y_{max}\ \text{score}$.
2. **Intersection sur Union :** Pour deux boîtes $A$ et $B$,
   $$
   \mathrm{IoU}(A,B) = \frac{\text{aire}(A \cap B)}{\text{aire}(A \cup B)},
   $$
   avec une aire d’intersection nulle lorsque les boîtes ne se chevauchent pas.
3. **Algorithme de NMS** (exactement comme décrit dans le chapitre) :
   
   a. Trier les boîtes par `score` décroissant (les égalités conservent l’ordre de lecture original).

   b. Sélectionner la boîte ayant le score le plus élevé parmi les restantes ; l’ajouter à la sortie et la retirer de la liste.

   c. Éliminer, de la liste restante, **toutes** les boîtes dont l’IoU avec la boîte sélectionnée est **supérieur ou égal** à $\tau$ — seules les boîtes avec $\mathrm{IoU} < \tau$ restent candidates.
   
   d. Répéter (b)–(c) jusqu’à ce que la liste des restantes soit vide.

4. **Sortie :** Pour chaque boîte conservée, dans l’ordre où elle a été sélectionnée, imprimer son indice original (position de lecture, à partir de $0$) et son `score`, avec 2 décimales. À la fin, imprimer `Total mantidas: X`.

#### 📌 Contraintes computationnelles

* **Attention au sens du seuil :** contrairement à ce que l’on pourrait supposer, une boîte est **supprimée** lorsque $\mathrm{IoU} \ge \tau$ (et non seulement lorsque $\mathrm{IoU} > \tau$) — suivez exactement ce critère, le même que celui du code de référence du chapitre.
* **Indices originaux :** la sortie référence la position de lecture de chaque boîte dans l’entrée, et non sa position après le tri par `score`.
* **Aire sans ajout de 1 pixel :** utilisez l’aire $= (x_{max}-x_{min}) \times (y_{max}-y_{min})$, exactement comme dans le chapitre (sans l’ajustement « +1 » parfois utilisé dans d’autres conventions).

#### 🧠 Fondements théoriques

| Élément | Rôle dans le post-traitement |
|---|---|
| IoU | Quantifie le chevauchement spatial entre deux boîtes englobantes |
| *Sliding window* (Haar Cascade) | Produit généralement plusieurs détections chevauchantes pour le même objet, à des positions et échelles proches |
| Seuil $\tau$ | Contrôle l’agressivité de la suppression : trop faible fusionne les objets proches ; trop élevé laisse passer les redondances |
| Tri par `score` | Garantit que, parmi les boîtes redondantes, celle avec la plus grande confiance survive toujours |

#### 📦 Spécification d’entrée et de sortie (VPL)

**Entrée :**

* Ligne 1 : Entier $N$ et réel $\tau$.
* Lignes suivantes $N$ : cinq réels $x_{min}\ y_{min}\ x_{max}\ y_{max}\ \text{score}$.

**Sortie :**

* Une ligne par boîte conservée, dans l’ordre de sélection : `indice score` (score avec 2 décimales).
* Dernière ligne : `Total mantidas: X`.

#### 📌 Exemples

| Entrée | Sortie | Observation |
|---|---|---|
| 5 0.4<br>50 50 150 150 0.90<br>60 55 155 145 0.75<br>58 60 160 150 0.60<br>300 300 400 420 0.95<br>310 305 395 415 0.70 | 3 0.95<br>0 0.90<br>Total mantidas: 2 | Exactement l’exemple de la figure du chapitre : 5 boîtes redondantes (2 objets) deviennent 2 détections finales. L’IoU entre la 1re et la 2e boîtes est $\approx 0{,}775$, bien au-dessus de $\tau=0{,}4$. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0804" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0804 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0804 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0804 button:hover { background: #e8dfcf; }
  #sim-ep0804 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0804_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0804_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulateur EP08_04 : IoU et suppression des non-maxima (NMS)</span>
  <span class="sim-ep0804_pill">Suppression si IoU &ge; &tau;</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0804_panel" style="margin-bottom:14px;">
    <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:12px;">
      
      <div>
        <div style="display:flex; justify-content:space-between; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Déplacement de la candidate (dx)</label>
          <span id="sim-ep0804_dx_v" style="font-family:monospace; font-weight:700; color:#26241d;">3</span>
        </div>
        <input id="sim-ep0804_dx" type="range" min="0" max="10" step="1" value="3">
      </div>

      <div>
        <div style="display:flex; justify-content:space-between; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Seuil (&tau;)</label>
          <span id="sim-ep0804_tau_v" style="font-family:monospace; font-weight:700; color:#26241d;">0.40</span>
        </div>
        <input id="sim-ep0804_tau" type="range" min="0.1" max="0.9" step="0.05" value="0.4">
      </div>

    </div>

    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      La boîte bleue (score plus élevé) a déjà été sélectionnée. Ajustez le chevauchement et le seuil &tau; pour vérifier la suppression de la boîte rouge (candidate).
    </div>
  </div>

  <!-- Canvas Visual de Caixas Delimitadoras -->
  <div class="sim-ep0804_panel" style="position:relative; width:100%; height:160px; margin-bottom:14px; overflow:hidden;">
    <div id="sim-ep0804_boxA" style="position:absolute; border:2px solid #2980b9; background:rgba(41,128,185,0.20); border-radius:4px; transition:all 0.15s ease;"></div>
    <div id="sim-ep0804_boxB" style="position:absolute; border:2px solid #c0392b; background:rgba(192,57,43,0.20); border-radius:4px; transition:all 0.15s ease;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0804_debug" class="sim-ep0804_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim08Ep04(root){
    if (!root || root.dataset.sim08Ep04Init) return;
    root.dataset.sim08Ep04Init = "1";

    var dxEl   = root.querySelector('#sim-ep0804_dx');
    var dxvEl  = root.querySelector('#sim-ep0804_dx_v');
    var tauEl  = root.querySelector('#sim-ep0804_tau');
    var tauvEl = root.querySelector('#sim-ep0804_tau_v');
    var boxA   = root.querySelector('#sim-ep0804_boxA');
    var boxB   = root.querySelector('#sim-ep0804_boxB');
    var dbg    = root.querySelector('#sim-ep0804_debug');

    var ESCALA = 10;
    var A = {x1: 5, y1: 3, x2: 15, y2: 13};

    function iou(a, b){
      var ix1 = Math.max(a.x1, b.x1), iy1 = Math.max(a.y1, b.y1);
      var ix2 = Math.min(a.x2, b.x2), iy2 = Math.min(a.y2, b.y2);
      var iw  = Math.max(0, ix2 - ix1), ih = Math.max(0, iy2 - iy1);
      var inter = iw * ih;
      var areaA = (a.x2 - a.x1) * (a.y2 - a.y1);
      var areaB = (b.x2 - b.x1) * (b.y2 - b.y1);
      return inter / (areaA + areaB - inter);
    }

    function render(){
      var dx  = parseInt(dxEl.value, 10);
      var tau = parseFloat(tauEl.value);

      dxvEl.textContent  = dx;
      tauvEl.textContent = tau.toFixed(2);

      var B = {x1: 5 + dx, y1: 3 + dx * 0.4, x2: 15 + dx, y2: 13 + dx * 0.4};

      boxA.style.left   = (A.x1 * ESCALA) + 'px';
      boxA.style.top    = (A.y1 * ESCALA) + 'px';
      boxA.style.width  = ((A.x2 - A.x1) * ESCALA) + 'px';
      boxA.style.height = ((A.y2 - A.y1) * ESCALA) + 'px';

      boxB.style.left   = (B.x1 * ESCALA) + 'px';
      boxB.style.top    = (B.y1 * ESCALA) + 'px';
      boxB.style.width  = ((B.x2 - B.x1) * ESCALA) + 'px';
      boxB.style.height = ((B.y2 - B.y1) * ESCALA) + 'px';

      var val = iou(A, B);
      var suprimida = val >= tau;

      if (suprimida) {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      } else {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      }

      dbg.textContent = 'IoU(A,B) = ' + val.toFixed(4) + '  |  \u03C4 = ' + tau.toFixed(2) + '  \u2192  Candidata (Vermelha) ' + 
        (suprimida ? 'SUPRIMIDA (IoU \u2265 \u03C4)' : 'MANTIDA (IoU < \u03C4)');
    }

    dxEl.addEventListener('input', render);
    tauEl.addEventListener('input', render);
    render();
  }

  function tryInitSim08Ep04(){
    var root = document.getElementById('sim-ep0804');
    if (root) initSim08Ep04(root); else setTimeout(tryInitSim08Ep04, 200);
  }
  tryInitSim08Ep04();
})();
</script>
""")

**Figure 8.4:** Simulateur EP08_04: IoU et suppression des non-maximums


In [ ]:
%%writefile EP08_04.py
# Code Python

In [ ]:
TestSuite("EP08_04.py").run()

### EP08_05 🟡 Étiquetage des Composantes Connexes : Segmentation d'Instances

L'exemple de segmentation classique de ce chapitre a séparé les « instances » de pièces simplement par leur déconnexion spatiale dans le masque binaire résultant du seuillage d'Otsu. Cette étape finale — étiqueter chaque composante connexe avec un identifiant d'instance — est exactement ce que vous êtes chargé d'implémenter ici, à partir de zéro, sur un masque binaire déjà prêt (0 = fond, 1 = objet), comme s'il s'agissait d'une réimplémentation manuelle de `cv2.connectedComponents`.

Cet exercice met également en évidence, de manière très concrète, la limitation discutée dans le chapitre : le résultat dépend entièrement de la façon dont on définit la « voisinage » entre pixels — et, comme vous le verrez dans le deuxième exemple, deux pixels en diagonale peuvent être considérés comme la même instance ou comme des instances différentes, en fonction exclusivement de la **connectivité** choisie, et non d'une quelconque notion sémantique d'objet.

#### 📋 Directives d'Implémentation

1. **Entrée :** Lire les dimensions $H \times W$ du masque binaire et ses $H \times W$ valeurs ($0$ ou $1$).
2. **Connectivité :** Lire l'entier $c \in \{4, 8\}$. En connectivité $4$, les voisins de $(i,j)$ sont $(i{-}1,j)$, $(i{+}1,j)$, $(i,j{-}1)$ et $(i,j{+}1)$. En connectivité $8$, on ajoute les quatre diagonales : $(i{-}1,j{-}1)$, $(i{-}1,j{+}1)$, $(i{+}1,j{-}1)$ et $(i{+}1,j{+}1)$.
3. **Découverte des composantes :** En parcourant le masque ligne par ligne, de gauche à droite et de haut en bas, chaque fois qu'un pixel de valeur $1$ encore non étiqueté est trouvé, il initie une **nouvelle composante** : attribuez-lui le prochain étiquette disponible (la première composante découverte reçoit l'étiquette $1$, la deuxième l'étiquette $2$, et ainsi de suite) et propagez cette même étiquette à tous les pixels de valeur $1$ atteignables à partir de lui par une chaîne de voisins (selon la connectivité choisie) — par recherche en largeur, en profondeur, ou par *union-find*, à votre choix.
4. **Pixels de fond :** restent avec l'étiquette $0$ et n'appartiennent à aucune instance.
5. **Sortie :** D'abord, imprimer la carte complète des étiquettes — $H$ lignes avec $W$ entiers chacune. Ensuite, pour chaque étiquette $\ell$ de $1$ à $K$ (dans l'ordre de découverte), imprimer `Instance l: A pixels`, où $A$ est la quantité de pixels avec cette étiquette. Enfin, imprimer `Total d'instances: K`.

#### 📌 Contraintes Computationnelles

* **Ordre de découverte = ordre de parcours :** les étiquettes sont numérotées dans l'ordre où chaque nouvelle composante est trouvée par le parcours ligne par ligne, et non par taille ou position.
* **Connectivité explicite :** deux pixels de valeur $1$ n'appartiennent à la même instance que s'il existe une chaîne de voisins **selon $c$** les reliant l'un à l'autre — ne pas utiliser par erreur la connectivité opposée.
* **Masque binaire pur :** toutes les valeurs d'entrée sont exactement $0$ ou $1$.

#### 🧠 Fondement Théorique

| Élément | Rôle dans la segmentation classique d'instances |
|---|---|
| Seuillage (Otsu, Chap. 4) | Étape précédente qui produit le masque binaire à partir de l'image d'intensité |
| Composante connexe | Chaque instance est définie **uniquement** par la connectivité spatiale des pixels d'objet, sans aucune notion de forme, de classe ou d'apparence |
| Connectivité 4 vs. 8 | Paramètre qui modifie le résultat : en connectivité 8, deux blobs unis uniquement en diagonale deviennent une seule instance |
| Limitation centrale | La technique fusionne des instances qui se touchent ou se chevauchent (même si ce sont des objets clairement distincts), car il n'y a pas de notion d'« objet » — seulement de « région connectée » |

#### 📦 Spécification d'Entrée et de Sortie (VPL)

**Entrée :**

* Ligne 1 : Entiers $H$ et $W$.
* Les $H$ lignes suivantes : $W$ entiers ($0$ ou $1$) chacune.
* Dernière ligne : Entier $c$ ($4$ ou $8$).

**Sortie :**

* $H$ lignes avec $W$ entiers chacune (la carte des étiquettes).
* Une ligne par instance, dans l'ordre de découverte : `Instance l: A pixels`.
* Dernière ligne : `Total d'instances: K`.

#### 📌 Exemples

| Entrée | Sortie | Observation |
|---|---|---|
| 6 6<br>0 0 0 0 0 0<br>0 1 1 0 0 0<br>0 1 1 0 0 0<br>0 0 0 0 0 0<br>0 0 0 0 1 1<br>0 0 0 0 1 1<br>8 | 0 0 0 0 0 0<br>0 1 1 0 0 0<br>0 1 1 0 0 0<br>0 0 0 0 0 0<br>0 0 0 0 2 2<br>0 0 0 0 2 2<br>Instance 1: 4 pixels<br>Instance 2: 4 pixels<br>Total d'instances: 2 | Deux blocs $2\times2$ clairement séparés : le résultat est le même en connectivité 4 ou 8. |
| 2 2<br>1 0<br>0 1<br>8 | 1 0<br>0 1<br>Instance 1: 2 pixels<br>Total d'instances: 1 | En connectivité 8, les deux pixels en diagonale appartiennent à la **même** instance. Répétez cet exemple avec $c=4$ : le résultat devient 2 instances de 1 pixel chacune — uniquement par le changement de connectivité, sans aucune différence dans le masque. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0805" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0805 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0805 button { font-size: 11px; padding: 6px 16px; border-radius: 20px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0805 button:hover { background: #e8dfcf; }
  #sim-ep0805 button.sim-ep0805_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0805_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0805_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulateur EP08_05 : Composants connectés (Connectivité 4 vs. 8)</span>
  <span class="sim-ep0805_pill">Même masque &rarr; Étiquettes différentes</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Controles de Seleção de Conectividade -->
  <div class="sim-ep0805_panel" style="margin-bottom:14px;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; text-align:center; margin-bottom:10px;">
      Le même masque (deux pixels en diagonale) &mdash; basculez la connectivité et observez le nombre d'instances et les couleurs des étiquettes changer.
    </div>

    <div style="display:flex; gap:8px; justify-content:center;">
      <button id="sim-ep0805_c4">Connectivité 4</button>
      <button id="sim-ep0805_c8" class="sim-ep0805_active">Connectivité 8</button>
    </div>
  </div>

  <!-- Exibição da Grade 2x2 -->
  <div class="sim-ep0805_panel" style="margin-bottom:14px; display:flex; justify-content:center;">
    <div id="sim-ep0805_grid" style="display:grid; grid-template-columns:repeat(2, 56px); gap:6px;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0805_debug" class="sim-ep0805_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim08Ep05(root){
    if (!root || root.dataset.sim08Ep05Init) return;
    root.dataset.sim08Ep05Init = "1";

    var mask = [[1, 0], [0, 1]];
    var conect = 8;
    var CORES = ['#eafaf1', '#fdecea'];
    var BORDAS = ['#a3e4d7', '#f5b7b1'];
    var TEXTOS = ['#04342C', '#c0392b'];

    var btn4   = root.querySelector('#sim-ep0805_c4');
    var btn8   = root.querySelector('#sim-ep0805_c8');
    var gridEl = root.querySelector('#sim-ep0805_grid');
    var dbg    = root.querySelector('#sim-ep0805_debug');

    function rotula(){
      var H = mask.length, W = mask[0].length;
      var labels = [[0, 0], [0, 0]];
      var atual = 0;
      var viz4 = [[-1, 0], [1, 0], [0, -1], [0, 1]];
      var viz8 = viz4.concat([[-1, -1], [-1, 1], [1, -1], [1, 1]]);
      var viz = conect === 8 ? viz8 : viz4;

      for (var i = 0; i < H; i++){
        for (var j = 0; j < W; j++){
          if (mask[i][j] === 1 && labels[i][j] === 0){
            atual++;
            var fila = [[i, j]];
            labels[i][j] = atual;
            while (fila.length){
              var pos = fila.pop();
              var r = pos[0], c = pos[1];
              for (var k = 0; k < viz.length; k++){
                var nr = r + viz[k][0], nc = c + viz[k][1];
                if (nr >= 0 && nr < H && nc >= 0 && nc < W && mask[nr][nc] === 1 && labels[nr][nc] === 0){
                  labels[nr][nc] = atual;
                  fila.push([nr, nc]);
                }
              }
            }
          }
        }
      }
      return {labels: labels, k: atual};
    }

    function estiloBotoes(){
      btn4.classList.toggle('sim-ep0805_active', conect === 4);
      btn8.classList.toggle('sim-ep0805_active', conect === 8);
    }

    function render(){
      var res = rotula();
      gridEl.innerHTML = '';

      for (var i = 0; i < 2; i++){
        for (var j = 0; j < 2; j++){
          var d = document.createElement('div');
          var lab = res.labels[i][j];
          var estilo = 'width:56px; height:56px; display:flex; align-items:center; justify-content:center; border-radius:8px; font-family:monospace; font-weight:700; font-size:13px; transition:all 0.15s ease;';
          
          if (lab === 0){
            estilo += 'background:#fafaf7; border:1px solid #e4dcc8; color:#8a8371;';
          } else {
            var idx = (lab - 1) % 2;
            estilo += 'background:' + CORES[idx] + '; border:2px solid ' + BORDAS[idx] + '; color:' + TEXTOS[idx] + ';';
          }

          d.style.cssText = estilo;
          d.textContent = mask[i][j] + (lab ? ' (r' + lab + ')' : '');
          gridEl.appendChild(d);
        }
      }

      estiloBotoes();

      if (res.k === 1) {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        dbg.style.borderColor = '#e4dcc8';
        dbg.style.background  = '#fafaf7';
        dbg.style.color       = '#26241d';
      }

      dbg.textContent = 'Connectivité = ' + conect + '  \u2192  ' + res.k + ' Instância(s) Encontrada(s)';
    }

    btn4.addEventListener('click', function(){ conect = 4; render(); });
    btn8.addEventListener('click', function(){ conect = 8; render(); });

    render();
  }

  function tryInitSim08Ep05(){
    var root = document.getElementById('sim-ep0805');
    if (root) initSim08Ep05(root); else setTimeout(tryInitSim08Ep05, 200);
  }
  tryInitSim08Ep05();
})();
</script>
""")

**Figure 8.5:** Simulateur EP08_05 : Étiquetage des composants connexes — Connectivité 4 vs. 8


In [ ]:
%%writefile EP08_05.py
# Code Python

In [ ]:
TestSuite("EP08_05.py").run()

### EP08_06 🟡 *Bounding Boxes*, Centroïdes et Propriétés des Instances avec `mm.measure`

Dans l'exercice précédent (**EP08_05**), on peut observer comment la segmentation par composantes connexes étiquette des régions binaires contiguës pour séparer les instances. Cependant, pour les tâches de détection, de suivi et d'analyse quantitative d'objets, la simple carte d'étiquettes ne suffit pas. Il devient nécessaire d'extraire des **métriques spatiales et géométriques** qui caractérisent chaque instance individuellement.

Cet EP se concentre sur le calcul et l'extraction automatique des propriétés fondamentales de vision par ordinateur pour chaque composante connexe trouvée dans le masque binaire, en utilisant la méthode native `mm.measure(img)` de la bibliothèque `morph` :

1. **Boîte englobante (*Bounding Box*) :** Le plus petit rectangle aligné sur les axes qui englobe complètement l'instance, défini par son coin supérieur gauche $(x, y)$, sa largeur $w$ et sa hauteur $h$.
2. **Centroïde géométrique $(\bar{x}, \bar{y})$ :** Le centre de masse de l'instance sur la grille discrète, équivalent aux moments spatiaux du premier ordre $M_{10}/M_{00}$ et $M_{01}/M_{00}$.
3. **Aire géométrique du contour ($A$) :** L'aire délimitée par le contour de l'instance calculée via `mm.contourArea(c)`.



#### 📋 Directives d'implémentation

1. **Entrée :** Lire les dimensions $H \times W$ du masque binaire, les $H \times W$ valeurs ($0$ ou $1$) et le paramètre de connectivité $c \in \{4, 8\}$.
2. **Extraction automatique avec `mm.measure` :** Passer l'image binarisée à la fonction `mm.measure(img_bin)`, qui extrait les contours OpenCV et retourne une liste de dictionnaires contenant les propriétés géométriques de chaque instance.
3. **Propriétés retournées :** Pour chaque dictionnaire $m$ de la liste retournée par `medidas = mm.measure(img_bin)` :
   * **Aire (`area`) :** Valeur numérique de l'aire géométrique du contour `mm.contourArea(c)`.
   * **Boîte englobante (`bbox`) :** Tuple $(x, y, w, h)$ représentant le coin supérieur gauche, la largeur et la hauteur.
   * **Centroïde (`center`) :** Tuple $(c_x, c_y)$ avec les coordonnées du centre de masse $M_{10}/M_{00}$ et $M_{01}/M_{00}$. Formater avec **deux décimales**.
4. **Sortie :** Pour chaque instance $1, \dots, K$ trouvée (triée par ordre de découverte/position dans l'image), imprimer une ligne contenant ses propriétés. Enfin, imprimer le nombre total d'instances.
   - Pour trier, utiliser `medidas.sort(key=lambda m: (m['bbox'][1], m['bbox'][0]))`.



#### 🧠 Fondements théoriques

| Propriété dans `mm.measure` | Calcul mathématique / Logique discrète | Application pratique en vision |
| --- | --- | --- |
| **`bbox` (OpenCV)** | $[x, y, w, h] = [\min(c), \min(r), \Delta c + 1, \Delta r + 1]$ | Format classique d'OpenCV. *Remarque : des réseaux comme YOLO convertissent ce rectangle en $(c_x, c_y, w, h)$ normalisé.* |
| **`center`** | $\bar{x} = \frac{M_{10}}{M_{00}}, \quad \bar{y} = \frac{M_{01}}{M_{00}}$ | Centre de masse exact du masque (utilisé pour le suivi et l'analyse de trajectoire). |
| **`area`** | $A = \text{contourArea}(C)$ (Formule du polygone) | Métrique continue de la surface de l'objet. |


#### 📦 Spécification d'entrée et de sortie (VPL)

**Entrée :**

* Ligne 1 : Entiers $H$ et $W$.
* $H$ lignes suivantes : $W$ entiers ($0$ ou $1$) chacun.
* Dernière ligne : Entier $c$ ($4$ ou $8$).

**Sortie :**

* Une ligne par instance dans l'ordre de découverte :
`Instância l: Area=A, BBox=(x,y,w,h), Centroide=(cx,cy)`
* Dernière ligne : `Total de instâncias: K`.



#### 📌 Exemples

| Entrée | Sortie | Observation |
|---|---|---|
| 6 6<br>0 0 0 0 0 0<br>0 1 1 0 0 0<br>0 1 1 0 0 0<br>0 0 0 0 0 0<br>0 0 0 0 1 1<br>0 0 0 0 1 1<br>8 | Instância 1: Area=1.0, BBox=(1,1,2,2), Centroide=(1.50,1.50)<br>Instância 2: Area=1.0, BBox=(4,4,2,2), Centroide=(4.50,4.50)<br>Total de instâncias: 2 | Blocs $2\times2$ alignés. Le calcul de l'aire géométrique du contour donne $1.0$. Le centroïde du bloc dans les colonnes 1–2 et les lignes 1–2 est exactement $(1.50,\,1.50)$. |
| 4 6<br>0 0 0 0 0 0<br>0 1 1 1 1 0<br>0 0 0 1 0 0<br>0 0 0 0 0 0<br>4 | Instância 1: Area=2.0, BBox=(1,1,4,2), Centroide=(2.40,1.20)<br>Total de instâncias: 1 | Objet asymétrique en forme de "T" inversé. L'aire géométrique du contour est $2.0$. Le centroïde reflète la distribution des pixels de l'objet. |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0806" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧮 Simulateur EP08_06 : Métriques morphologiques natives (mm.measure)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">Contour OpenCV & Moments</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <div style="display:flex;justify-content:space-between;align-items:flex-end;margin-bottom:14px;flex-wrap:wrap;gap:10px;">
      <div>
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">ACTION</div>
        <button id="ep0806_btnRandom" style="background:#26241d;color:#7ee7c6;border:none;padding:8px 14px;font-size:11px;font-weight:700;border-radius:9px;cursor:pointer;display:inline-flex;align-items:center;gap:6px;font-family:monospace;">🎲 Générer de nouvelles instances binaires</button>
      </div>
      <div style="font-size:11px;color:#8a8371;font-family:monospace;">
        <span style="font-weight:700;color:#26241d;">Paramètre de précision (approxPolyDP) :</span> precision = 0.01
      </div>
    </div>

    <!-- Container da Matriz de Píxeis -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;margin-bottom:14px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:8px;text-align:left;">CARTE DES ÉTIQUETTES D'INSTANCES</div>
      <div id="ep0806_grid_container" style="overflow-x:auto;padding-bottom:4px;"></div>
    </div>

    <!-- Tabela de Métricas do mm.measure -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">MÉTRIQUES EXTRAITES PAR MM.MEASURE</div>
      <div style="overflow-x:auto;">
        <table style="width:100%;border-collapse:collapse;font-size:10.5px;font-family:monospace;margin-top:10px;">
          <thead>
            <tr style="background:#f1ead7;">
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">id</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">aire</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">périmètre</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">centre (cx, cy)</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">bBox (x,y,w,h)</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">circularité</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">solidité</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">sommets</th>
            </tr>
          </thead>
          <tbody id="ep0806_tbody"></tbody>
        </table>
      </div>
    </div>
  </div>
</div>

<script>
(function(){
  function initSim(root){
    if (!root || root.dataset.ep0806Init) return;
    root.dataset.ep0806Init = "1";

    var H = 10, W = 22;
    var mask = [], labels = [], metrics = [];
    var colors = ['#ffffff', '#7ee7c6', '#fca5a5', '#fde047', '#93c5fd', '#c084fc', '#f472b6'];

    function cv2_findContours(mat) {
      var visited = Array.from({length: H}, function(){ return Array(W).fill(false); });
      var contours = [];

      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          if (mat[r][c] === 1 && !visited[r][c]) {
            var blobPixels = [];
            var queue = [[r, c]];
            visited[r][c] = true;

            while (queue.length > 0) {
              var curr = queue.shift();
              var cr = curr[0], cc = curr[1];
              blobPixels.push([cr, cc]);

              var dirs = [[-1,0],[1,0],[0,-1],[0,1],[-1,-1],[-1,1],[1,-1],[1,1]];
              for (var d = 0; d < dirs.length; d++) {
                var nr = cr + dirs[d][0], nc = cc + dirs[d][1];
                if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
                  if (mat[nr][nc] === 1 && !visited[nr][nc]) {
                    visited[nr][nc] = true;
                    queue.push([nr, nc]);
                  }
                }
              }
            }

            var borderPts = [];
            blobPixels.forEach(function(p) {
              var pr = p[0], pc = p[1];
              if (pr === 0 || pr === H-1 || pc === 0 || pc === W-1 ||
                  mat[pr-1][pc] === 0 || mat[pr+1][pc] === 0 ||
                  mat[pr][pc-1] === 0 || mat[pr][pc+1] === 0) {
                borderPts.push([pc, pr]);
              }
            });

            var cx = 0, cy = 0;
            borderPts.forEach(function(pt){ cx += pt[0]; cy += pt[1]; });
            cx /= borderPts.length; cy /= borderPts.length;

            borderPts.sort(function(a, b) {
              return Math.atan2(a[1] - cy, a[0] - cx) - Math.atan2(b[1] - cy, b[0] - cx);
            });

            contours.push({ pixels: blobPixels, contour: borderPts });
          }
        }
      }
      return contours;
    }

    function cv2_contourArea(contour) {
      if (contour.length < 3) return 0.0;
      var area = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        area += contour[i][0] * contour[j][1];
        area -= contour[j][0] * contour[i][1];
      }
      return Math.abs(area) / 2.0;
    }

    function cv2_arcLength(contour) {
      if (contour.length < 2) return 0.0;
      var per = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        per += Math.hypot(contour[j][0] - contour[i][0], contour[j][1] - contour[i][1]);
      }
      return per;
    }

    function cv2_moments(blobPixels) {
      var m00 = blobPixels.length;
      var m10 = 0.0, m01 = 0.0;
      blobPixels.forEach(function(p) { m10 += p[1]; m01 += p[0]; });
      return { m00: m00, cx: m00 > 0 ? m10 / m00 : 0, cy: m00 > 0 ? m01 / m00 : 0 };
    }

    function cv2_boundingRect(blobPixels) {
      var minX = W, maxX = 0, minY = H, maxY = 0;
      blobPixels.forEach(function(p) {
        var r = p[0], c = p[1];
        if (c < minX) minX = c; if (c > maxX) maxX = c;
        if (r < minY) minY = r; if (r > maxY) maxY = r;
      });
      return { x: minX, y: minY, w: maxX - minX + 1, h: maxY - minY + 1 };
    }

    function cv2_convexHull(points) {
      if (points.length <= 2) return points;
      var pts = points.slice().sort(function(a,b){ return a[0] === b[0] ? a[1] - b[1] : a[0] - b[0]; });
      function cross(o, a, b) { return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0]); }

      var lower = [];
      for (var i = 0; i < pts.length; i++) {
        while (lower.length >= 2 && cross(lower[lower.length - 2], lower[lower.length - 1], pts[i]) <= 0) lower.pop();
        lower.push(pts[i]);
      }
      var upper = [];
      for (var i = pts.length - 1; i >= 0; i--) {
        while (upper.length >= 2 && cross(upper[upper.length - 2], upper[upper.length - 1], pts[i]) <= 0) upper.pop();
        upper.push(pts[i]);
      }
      upper.pop(); lower.pop();
      return lower.concat(upper);
    }

    function cv2_approxPolyDP(contour, precision) {
      if (contour.length <= 2) return contour;
      var epsilon = precision * cv2_arcLength(contour);
      function rdp(pts, eps) {
        if (pts.length <= 2) return pts;
        var dmax = 0, index = 0, end = pts.length - 1;
        for (var i = 1; i < end; i++) {
          var dx = pts[end][0] - pts[0][0], dy = pts[end][1] - pts[0][1];
          var mag = Math.hypot(dx, dy);
          var d = mag === 0 ? Math.hypot(pts[i][0] - pts[0][0], pts[i][1] - pts[0][1]) :
            Math.abs(dy * pts[i][0] - dx * pts[i][1] + pts[end][0] * pts[0][1] - pts[end][1] * pts[0][0]) / mag;
          if (d > dmax) { index = i; dmax = d; }
        }
        if (dmax > eps) {
          var r1 = rdp(pts.slice(0, index + 1), eps);
          var r2 = rdp(pts.slice(index, end + 1), eps);
          return r1.slice(0, r1.length - 1).concat(r2);
        } else {
          return [pts[0], pts[end]];
        }
      }
      return rdp(contour, epsilon);
    }

    function measureJS(mat) {
      var blobs = cv2_findContours(mat);
      var res = [];
      labels = Array.from({length: H}, function(){ return Array(W).fill(0); });

      blobs.forEach(function(item, idx) {
        var contour = item.contour;
        var pixels = item.pixels;
        var labelId = idx + 1;

        pixels.forEach(function(p){ labels[p[0]][p[1]] = labelId; });

        var area = cv2_contourArea(contour);
        if (area === 0) area = pixels.length;

        var per = cv2_arcLength(contour);
        var bbox = cv2_boundingRect(pixels);
        var moments = cv2_moments(pixels);

        var hull = cv2_convexHull(contour);
        var hull_area = cv2_contourArea(hull);
        if (hull_area === 0) hull_area = area;

        var poly = cv2_approxPolyDP(contour, 0.01);

        res.push({
          id: labelId,
          area: area,
          perimeter: per,
          cx: moments.cx,
          cy: moments.cy,
          x: bbox.x, y: bbox.y, w: bbox.w, h: bbox.h,
          circularity: per > 0 ? (4 * Math.PI * area) / (per * per) : 0,
          solidity: hull_area > 0 ? area / hull_area : 0,
          vertices: poly.length
        });
      });

      res.sort(function(a, b) {
        if (a.y !== b.y) return a.y - b.y;
        return a.x - b.x;
      });

      res.forEach(function(m, i) { m.id = i + 1; });
      return res;
    }

    function render() {
      var gridContainer = root.querySelector('#ep0806_grid_container');
      gridContainer.innerHTML = '';
      var grid = document.createElement('div');
      grid.style.cssText = 'display:inline-grid;gap:1px;background:#e4dcc8;padding:1px;border-radius:6px;overflow:auto;max-width:100%;grid-template-columns:repeat(' + W + ', 22px);';

      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          var l = labels[r][c];
          var cell = document.createElement('div');
          var bg = l === 0 ? '#ffffff' : colors[(l % (colors.length - 1)) + 1];
          var fg = l === 0 ? '#8a8371' : '#26241d';
          cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:700;font-family:monospace;user-select:none;background:' + bg + ';color:' + fg + ';';
          cell.textContent = l;
          grid.appendChild(cell);
        }
      }
      gridContainer.appendChild(grid);

      var tbody = root.querySelector('#ep0806_tbody');
      tbody.innerHTML = '';

      if (metrics.length === 0) {
        tbody.innerHTML = '<tr><td colspan="8" style="padding:12px;color:#8a8371;text-align:center;">Nenhuma instância binária encontrada.</td></tr>';
        return;
      }

      metrics.forEach(function(m, i){
        var tr = document.createElement('tr');
        if (i % 2 === 1) tr.style.background = '#fafaf7';
        var centerStr = '(' + m.cx.toFixed(2) + ', ' + m.cy.toFixed(2) + ')';
        var bboxStr = '(' + m.x + ', ' + m.y + ', ' + m.w + ', ' + m.h + ')';

        tr.innerHTML = 
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;"><b>' + m.id + '</b></td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.area.toFixed(1) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.perimeter.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + centerStr + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + bboxStr + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.circularity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.solidity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.vertices + '</td>';
        tbody.appendChild(tr);
      });
    }

    function generate() {
      mask = Array.from({length: H}, function(){ return Array(W).fill(0); });
      var numObj = Math.floor(Math.random() * 2) + 2;

      for (var o = 0; o < numObj; o++) {
        var w = Math.floor(Math.random() * 3) + 3;
        var h = Math.floor(Math.random() * 3) + 3;
        var sr = Math.floor(Math.random() * (H - h));
        var sc = Math.floor(Math.random() * (W / numObj - w)) + Math.floor(o * (W / numObj));

        for (var r = 0; r < h; r++) {
          for (var c = 0; c < w; c++) {
            if (Math.random() > 0.15) mask[sr + r][sc + c] = 1;
          }
        }
      }

      metrics = measureJS(mask);
      render();
    }

    root.querySelector('#ep0806_btnRandom').addEventListener('click', generate);
    generate();
  }

  function tryInit(){
    var root = document.getElementById('sim-ep0806');
    if (root) initSim(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figure 8.6:** Simulador EP08_06 : Extraction de *Bounding Boxes*, Centroides et Propriétés avec *mm.measure*


In [ ]:
%%writefile EP08_06.py
# Code Python

In [ ]:
TestSuite("EP08_06.py").run()

### EP08_07 🟡 Suppression du Bruit Sel et Poivre et Mesure d'Objets

Dans cet exercice, vous appliquerez un filtrage morphologique pour nettoyer une image binaire corrompue par un bruit de type **sel et poivre** (pixels isolés de valeur `1` dans le fond et `0` à l'intérieur des objets). Après le nettoyage, le programme doit extraire les mesures géométriques des composants connectés restants, les trier et afficher le tableau final de métriques.

#### 📋 Directives d'Implémentation

1. **Entrée :** lire deux entiers $H$ et $W$ (hauteur et largeur de l'image) sur la première ligne, puis les $H$ lignes avec la matrice binaire contenant des pixels `0` et `1` séparés par des espaces.

2. **Filtrage Morphologique :** appliquer un enchaînement d'**Ouverture** (pour éliminer le bruit de sel dans le fond) suivi d'une **Fermeture** (pour combler le bruit de poivre à l'intérieur des objets) avec un élément structurant $3 \times 3$.

3. **Affichage de l'Image Nettoyée :** imprimer la matrice résultante en valeurs `0` et `1` séparées par des espaces.

4. **Mesures Géométriques :** pour chaque objet identifié dans la matrice nettoyée, extraire :
* `id` : identifiant numérique séquentiel (réattribué après le tri) ;
* `area` : aire calculée via le contour (`cv2.contourArea`) ;
* `perimeter` : périmètre du contour (`cv2.arcLength`) ;
* `cx`, `cy` : centre de masse (centroïde via `cv2.moments`) ;
* `x`, `y`, `w`, `h` : coordonnées du rectangle englobant (`cv2.boundingRect`) ;
* `circularity` : circularité donnée par $\frac{4 \pi \cdot \text{aire}}{\text{périmètre}^2}$ ;
* `solidity` : solidité donnée par le rapport $\frac{\text{aire}}{\text{aire de l'enveloppe convexe}}$ ;
* `vertices` : nombre de sommets approximé du polygone (`cv2.approxPolyDP` avec $\epsilon = 0.02 \times \text{périmètre}$).

5. **Tri et Sortie :** trier les objets par ordre croissant selon la position $X$ du rectangle englobant (`bbox[0]`) ; en cas d'égalité, utiliser la position $Y$ (`bbox[1]`). Réattribuer les `id`s de $1$ à $N$ et imprimer le tableau formaté.
   - Pour trier, utiliser `medidas.sort(key=lambda m: (m['bbox'][1], m['bbox'][0]))`, avec `medidas = mm.measure(img)`.

#### 📌 Contraintes et Règles de Tri

* **Règle de Tri des Objets :**
```python
medidas.sort(key=lambda m: (m['bbox'][0], m['bbox'][1]))

```

* **Différence d'Aire :** L'aire calculée par OpenCV (`cv2.contourArea`) mesure l'aire du polygone continu délimité par les centres des pixels de bordure, ce qui donne des valeurs numériques inférieures au simple comptage discret des pixels `1` (`np.sum`).

#### 🧠 Fondements Théoriques

| Opération / Métrique | Fonction dans le Filtrage et la Caractérisation |
|--------------------|---------------------------------------|
| **Ouverture Morphologique** ($\circ$) | Érosion suivie d'une dilatation : supprime les bruits brillants isolés (*sel*). |
| **Fermeture Morphologique** ($\bullet$) | Dilatation suivie d'une érosion : comble les petits trous sombres à l'intérieur des objets (*poivre*). |
| **`cv2.boundingRect`** | Retourne $(x, y, w, h)$, le plus petit rectangle aligné sur les axes qui englobe l'objet. |
| **Circularité et Solidité** | Décrivent la compacité et la convexité géométrique du composant. |

#### 📌 Exemples

| Entrée | Sortie |
|---|---|
| 8 9<br>0 0 0 0 0 0 0 0 0<br>0 0 0 1 1 1 1 0 0<br>0 0 0 1 1 1 1 0 0<br>0 0 0 1 1 1 1 0 0<br>0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 1 1<br>0 0 0 0 0 0 0 1 1<br>0 0 0 0 0 0 0 0 0 | id area perimeter cx cy x y w h circularity solidity vertices<br>1 9.0 12.0 3.5 2.0 3 1 4 3 0.79 1.000 4<br>2 4.0 8.0 7.5 5.5 7 5 2 2 0.79 1.000 4 |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0807" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧮 Simulateur EP08_07 : Morphologie Commutable (4-C / 8-C) & Métriques OpenCV</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">Sel + Poivre &rarr; Ouverture &rarr; Fermeture &rarr; Mesure</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:flex-end;margin-bottom:14px;">
      <div style="flex:2;min-width:260px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">ÉTAPE DU TRAITEMENT MORPHOLOGIQUE</div>
        <div style="display:flex;gap:4px;background:#f1ead7;border:1px solid #e4dcc8;border-radius:11px;padding:3px;">
          <button id="ep0807_btnOrig" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:700;border:none;background:#26241d;color:#fbf7ee;cursor:pointer;border-radius:9px;white-space:nowrap;">1. Bruitée</button>
          <button id="ep0807_btnAbert" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">2. Ouverture</button>
          <button id="ep0807_btnFech" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">3. Fermeture</button>
        </div>
      </div>

      <div style="flex:1;min-width:140px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">ÉLÉMENT STRUCTURANT</div>
        <div style="display:flex;gap:4px;background:#f1ead7;border:1px solid #e4dcc8;border-radius:11px;padding:3px;">
          <button id="ep0807_btnConn4" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:700;border:none;background:#26241d;color:#fbf7ee;cursor:pointer;border-radius:9px;white-space:nowrap;">4-Connexe</button>
          <button id="ep0807_btnConn8" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">8-Connexe</button>
        </div>
      </div>

      <div style="min-width:140px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">AFFICHAGE DES PIXELS</div>
        <div style="display:flex;gap:4px;background:#f1ead7;border:1px solid #e4dcc8;border-radius:11px;padding:3px;">
          <button id="ep0807_btnVal" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:700;border:none;background:#26241d;color:#fbf7ee;cursor:pointer;border-radius:9px;white-space:nowrap;">Valeurs (0/1)</button>
          <button id="ep0807_btnCor" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">Couleurs (P&W)</button>
        </div>
      </div>

      <div>
        <button id="ep0807_btnRandom" style="background:#26241d;color:#7ee7c6;border:none;padding:8px 14px;font-size:11px;font-weight:700;border-radius:9px;cursor:pointer;display:inline-flex;align-items:center;gap:6px;font-family:monospace;">🎲 Générer un Scénario Aléatoire</button>
      </div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;margin-bottom:14px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:8px;text-align:left;">VISUALISATION DE LA MATRICE DE PIXELS D'ENTRÉE / TRAITÉE</div>
      <div id="ep0807_grid_container" style="overflow-x:auto;padding-bottom:4px;"></div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">TABLEAU DES MESURES DES OBJETS (CALCULÉ APRÈS OUVERTURE ET FERMETURE)</div>
      <div style="overflow-x:auto;">
        <table style="width:100%;border-collapse:collapse;font-size:10.5px;font-family:monospace;margin-top:10px;">
          <thead>
            <tr style="background:#f1ead7;">
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">id</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">aire</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">périmètre</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cx</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cy</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">x</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">y</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">w</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">h</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">circularité</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">solidité</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">sommets</th>
            </tr>
          </thead>
          <tbody id="ep0807_tbody"></tbody>
        </table>
      </div>
    </div>
  </div>
</div>

<script>
(function(){
  function initSim(root){
    if (!root || root.dataset.ep0807Init) return;
    root.dataset.ep0807Init = "1";

    var H = 12, W = 28;
    var imgBase = [], imgRuido = [], imgAbertura = [], imgLimpa = [];
    var medidasObjetos = [];
    var etapaAtual = 'ruido', modoExibicao = 'val', modoConectividade = 4;

    var elBtnOrig = root.querySelector('#ep0807_btnOrig');
    var elBtnAbert = root.querySelector('#ep0807_btnAbert');
    var elBtnFech = root.querySelector('#ep0807_btnFech');
    var elBtnConn4 = root.querySelector('#ep0807_btnConn4');
    var elBtnConn8 = root.querySelector('#ep0807_btnConn8');
    var elBtnVal = root.querySelector('#ep0807_btnVal');
    var elBtnCor = root.querySelector('#ep0807_btnCor');
    var elBtnRandom = root.querySelector('#ep0807_btnRandom');
    var elGridContainer = root.querySelector('#ep0807_grid_container');
    var elTbody = root.querySelector('#ep0807_tbody');

    var neighbors4 = [[0,0], [-1,0], [1,0], [0,-1], [0,1]];
    var neighbors8 = [[0,0], [-1,0], [1,0], [0,-1], [0,1], [-1,-1], [-1,1], [1,-1], [1,1]];

    function dilate(mat, conn) {
      var neighbors = conn === 8 ? neighbors8 : neighbors4;
      var res = Array.from({length: H}, function(){ return Array(W).fill(0); });
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          var hit = false;
          for (var i = 0; i < neighbors.length; i++) {
            var nr = r + neighbors[i][0], nc = c + neighbors[i][1];
            if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
              if (mat[nr][nc] === 1) { hit = true; break; }
            }
          }
          res[r][c] = hit ? 1 : 0;
        }
      }
      return res;
    }

    function erode(mat, conn) {
      var neighbors = conn === 8 ? neighbors8 : neighbors4;
      var res = Array.from({length: H}, function(){ return Array(W).fill(0); });
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          var fit = true;
          for (var i = 0; i < neighbors.length; i++) {
            var nr = r + neighbors[i][0], nc = c + neighbors[i][1];
            if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
              if (mat[nr][nc] !== 1) { fit = false; break; }
            } else { fit = false; }
          }
          res[r][c] = fit ? 1 : 0;
        }
      }
      return res;
    }

    function cv2_findContours(mat) {
      var visited = Array.from({length: H}, function(){ return Array(W).fill(false); });
      var contours = [];
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          if (mat[r][c] === 1 && !visited[r][c]) {
            var blobPixels = [], queue = [[r, c]];
            visited[r][c] = true;
            while (queue.length > 0) {
              var curr = queue.shift(), cr = curr[0], cc = curr[1];
              blobPixels.push([cr, cc]);
              var dirs = [[-1,0],[1,0],[0,-1],[0,1],[-1,-1],[-1,1],[1,-1],[1,1]];
              for (var d = 0; d < dirs.length; d++) {
                var nr = cr + dirs[d][0], nc = cc + dirs[d][1];
                if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
                  if (mat[nr][nc] === 1 && !visited[nr][nc]) {
                    visited[nr][nc] = true; queue.push([nr, nc]);
                  }
                }
              }
            }
            var borderPts = [];
            blobPixels.forEach(function(p) {
              var pr = p[0], pc = p[1];
              if (pr === 0 || pr === H-1 || pc === 0 || pc === W-1 ||
                  mat[pr-1][pc] === 0 || mat[pr+1][pc] === 0 ||
                  mat[pr][pc-1] === 0 || mat[pr][pc+1] === 0) { borderPts.push([pc, pr]); }
            });
            var cx = 0, cy = 0;
            borderPts.forEach(function(pt){ cx += pt[0]; cy += pt[1]; });
            cx /= borderPts.length; cy /= borderPts.length;
            borderPts.sort(function(a, b) { return Math.atan2(a[1] - cy, a[0] - cx) - Math.atan2(b[1] - cy, b[0] - cx); });
            contours.push({ pixels: blobPixels, contour: borderPts });
          }
        }
      }
      return contours;
    }

    function cv2_contourArea(contour) {
      if (contour.length < 3) return 0.0;
      var area = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        area += contour[i][0] * contour[j][1] - contour[j][0] * contour[i][1];
      }
      return Math.abs(area) / 2.0;
    }

    function cv2_arcLength(contour) {
      if (contour.length < 2) return 0.0;
      var per = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        per += Math.hypot(contour[j][0] - contour[i][0], contour[j][1] - contour[i][1]);
      }
      return per;
    }

    function cv2_moments(blobPixels) {
      var m00 = blobPixels.length, m10 = 0.0, m01 = 0.0;
      blobPixels.forEach(function(p) { m10 += p[1]; m01 += p[0]; });
      return { m00: m00, cx: m00 > 0 ? m10 / m00 : 0, cy: m00 > 0 ? m01 / m00 : 0 };
    }

    function cv2_boundingRect(blobPixels) {
      var minX = W, maxX = 0, minY = H, maxY = 0;
      blobPixels.forEach(function(p) {
        var r = p[0], c = p[1];
        if (c < minX) minX = c; if (c > maxX) maxX = c;
        if (r < minY) minY = r; if (r > maxY) maxY = r;
      });
      return { x: minX, y: minY, w: maxX - minX + 1, h: maxY - minY + 1 };
    }

    function cv2_convexHull(points) {
      if (points.length <= 2) return points;
      var pts = points.slice().sort(function(a,b){ return a[0] === b[0] ? a[1] - b[1] : a[0] - b[0]; });
      function cross(o, a, b) { return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0]); }
      var lower = [];
      for (var i = 0; i < pts.length; i++) {
        while (lower.length >= 2 && cross(lower[lower.length - 2], lower[lower.length - 1], pts[i]) <= 0) lower.pop();
        lower.push(pts[i]);
      }
      var upper = [];
      for (var i = pts.length - 1; i >= 0; i--) {
        while (upper.length >= 2 && cross(upper[upper.length - 2], upper[upper.length - 1], pts[i]) <= 0) upper.pop();
        upper.push(pts[i]);
      }
      upper.pop(); lower.pop();
      return lower.concat(upper);
    }

    function cv2_approxPolyDP(contour, precision) {
      if (contour.length <= 2) return contour;
      var epsilon = precision * cv2_arcLength(contour);
      function rdp(pts, eps) {
        if (pts.length <= 2) return pts;
        var dmax = 0, index = 0, end = pts.length - 1;
        for (var i = 1; i < end; i++) {
          var dx = pts[end][0] - pts[0][0], dy = pts[end][1] - pts[0][1];
          var mag = Math.hypot(dx, dy);
          var d = mag === 0 ? Math.hypot(pts[i][0] - pts[0][0], pts[i][1] - pts[0][1]) :
            Math.abs(dy * pts[i][0] - dx * pts[i][1] + pts[end][0] * pts[0][1] - pts[end][1] * pts[0][0]) / mag;
          if (d > dmax) { index = i; dmax = d; }
        }
        if (dmax > eps) {
          var r1 = rdp(pts.slice(0, index + 1), eps);
          var r2 = rdp(pts.slice(index, end + 1), eps);
          return r1.slice(0, r1.length - 1).concat(r2);
        } else { return [pts[0], pts[end]]; }
      }
      return rdp(contour, epsilon);
    }

    function measureOpenCV(mat) {
      var blobs = cv2_findContours(mat);
      var medidas = [];
      blobs.forEach(function(item) {
        var contour = item.contour, pixels = item.pixels;
        var area = cv2_contourArea(contour);
        if (area === 0) area = pixels.length;
        var per = cv2_arcLength(contour);
        var bbox = cv2_boundingRect(pixels);
        var moments = cv2_moments(pixels);
        var hull = cv2_convexHull(contour);
        var hull_area = cv2_contourArea(hull);
        if (hull_area === 0) hull_area = area;
        var poly = cv2_approxPolyDP(contour, 0.01);
        medidas.push({
          area: area, perimeter: per, cx: moments.cx, cy: moments.cy,
          x: bbox.x, y: bbox.y, w: bbox.w, h: bbox.h,
          circularity: per > 0 ? (4 * Math.PI * area) / (per * per) : 0,
          solidity: hull_area > 0 ? area / hull_area : 0, vertices: poly.length
        });
      });
      medidas.sort(function(a, b) { return a.x !== b.x ? a.x - b.x : a.y - b.y; });
      medidas.forEach(function(m, i) { m.id = i + 1; });
      return medidas;
    }

    function recalcularMorfologia() {
      imgAbertura = dilate(erode(imgRuido, modoConectividade), modoConectividade);
      imgLimpa = erode(dilate(imgAbertura, modoConectividade), modoConectividade);
      medidasObjetos = measureOpenCV(imgLimpa);
      renderGrid();
      renderTabela();
    }

    function gerarCenarioAleatorio() {
      imgBase = Array.from({length: H}, function(){ return Array(W).fill(0); });
      var numObjetos = Math.floor(Math.random() * 2) + 2; 
      var setores = [ { minC: 1, maxC: 8 }, { minC: 10, maxC: 17 }, { minC: 19, maxC: 25 } ];
      var objetoPixels = [];
      for (var o = 0; o < numObjetos; o++) {
        var setor = setores[o];
        var tipoForma = Math.floor(Math.random() * 3);
        var objW = Math.floor(Math.random() * 2) + 4, objH = Math.floor(Math.random() * 2) + 4;
        var startC = Math.floor(Math.random() * (setor.maxC - setor.minC - objW + 1)) + setor.minC;
        var startR = Math.floor(Math.random() * (H - 4 - objH + 1)) + 2;
        for (var r = 0; r < objH; r++) {
          for (var c = 0; c < objW; c++) {
            var pr = startR + r, pc = startC + c, isObj = false;
            if (tipoForma === 0) isObj = true;
            else if (tipoForma === 1) { if (r >= objH / 2 || c < objW / 2) isObj = true; }
            else if (tipoForma === 2) { if (r < objH / 2 || (c >= Math.floor(objW / 3) && c <= Math.floor(2 * objW / 3))) isObj = true; }
            if (isObj) { imgBase[pr][pc] = 1; objetoPixels.push([pr, pc]); }
          }
        }
      }
      imgRuido = JSON.parse(JSON.stringify(imgBase));
      var qtdSal = Math.floor(Math.random() * 2) + 2;
      for (var s = 0; s < qtdSal; s++) {
        var sr = Math.floor(Math.random() * (H - 2)) + 1, sc = Math.floor(Math.random() * (W - 2)) + 1;
        if (imgBase[sr][sc] === 0) imgRuido[sr][sc] = 1;
      }
      var shuffledObj = objetoPixels.filter(function(p){ return p[0] > 0 && p[0] < H-1 && p[1] > 0 && p[1] < W-1; }).sort(function() { return 0.5 - Math.random(); });
      var qtdPimenta = Math.max(1, Math.floor(shuffledObj.length * 0.10));
      for (var p = 0; p < qtdPimenta; p++) { imgRuido[shuffledObj[p][0]][shuffledObj[p][1]] = 0; }
      recalcularMorfologia();
    }

    function renderGrid(){
      var mat = imgRuido;
      if (etapaAtual === 'abertura') mat = imgAbertura;
      if (etapaAtual === 'fechamento') mat = imgLimpa;
      elGridContainer.innerHTML = '';
      var grid = document.createElement('div');
      grid.style.cssText = 'display:inline-grid;gap:1px;background:#e4dcc8;padding:1px;border-radius:6px;overflow:auto;max-width:100%;grid-template-columns:repeat(' + W + ', 22px);';
      for (var r = 0; r < H; r++){
        for (var c = 0; c < W; c++){
          var val = mat[r][c], cell = document.createElement('div');
          var bg = val === 1 ? '#26241d' : '#ffffff';
          var fg = val === 1 ? '#7ee7c6' : '#8a8371';
          cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:700;font-family:monospace;user-select:none;background:' + bg + ';color:' + fg + ';';
          if (modoExibicao === 'val') cell.textContent = val; else cell.textContent = '';
          grid.appendChild(cell);
        }
      }
      elGridContainer.appendChild(grid);
    }

    function renderTabela(){
      elTbody.innerHTML = '';
      if (medidasObjetos.length === 0) {
        elTbody.innerHTML = '<tr><td colspan="12" style="padding:12px;color:#8a8371;text-align:center;">Nenhum objeto detectado após o processamento morfológico.</td></tr>';
        return;
      }
      medidasObjetos.forEach(function(m, i){
        var tr = document.createElement('tr');
        if (i % 2 === 1) tr.style.background = '#fafaf7';
        tr.innerHTML = 
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;"><b>' + m.id + '</b></td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.area.toFixed(1) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.perimeter.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cx.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cy.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.x + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.y + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.w + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.h + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.circularity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.solidity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.vertices + '</td>';
        elTbody.appendChild(tr);
      });
    }

    function setEtapa(etapa, btn){
      [elBtnOrig, elBtnAbert, elBtnFech].forEach(function(b){ b.style.background = 'transparent'; b.style.color = '#8a8371'; b.style.fontWeight = '600'; });
      btn.style.background = '#26241d'; btn.style.color = '#fbf7ee'; btn.style.fontWeight = '700';
      etapaAtual = etapa; renderGrid();
    }

    function setConectividade(conn, btn){
      [elBtnConn4, elBtnConn8].forEach(function(b){ b.style.background = 'transparent'; b.style.color = '#8a8371'; b.style.fontWeight = '600'; });
      btn.style.background = '#26241d'; btn.style.color = '#fbf7ee'; btn.style.fontWeight = '700';
      modoConectividade = conn; recalcularMorfologia();
    }

    function setModo(modo, btn){
      [elBtnVal, elBtnCor].forEach(function(b){ b.style.background = 'transparent'; b.style.color = '#8a8371'; b.style.fontWeight = '600'; });
      btn.style.background = '#26241d'; btn.style.color = '#fbf7ee'; btn.style.fontWeight = '700';
      modoExibicao = modo; renderGrid();
    }

    elBtnOrig.addEventListener('click', function(){ setEtapa('ruido', elBtnOrig); });
    elBtnAbert.addEventListener('click', function(){ setEtapa('abertura', elBtnAbert); });
    elBtnFech.addEventListener('click', function(){ setEtapa('fechamento', elBtnFech); });
    elBtnConn4.addEventListener('click', function(){ setConectividade(4, elBtnConn4); });
    elBtnConn8.addEventListener('click', function(){ setConectividade(8, elBtnConn8); });
    elBtnVal.addEventListener('click', function(){ setModo('val', elBtnVal); });
    elBtnCor.addEventListener('click', function(){ setModo('cor', elBtnCor); });
    elBtnRandom.addEventListener('click', function(){ gerarCenarioAleatorio(); });

    gerarCenarioAleatorio();
  }

  function tryInit(){
    var root = document.getElementById('sim-ep0807');
    if (root) initSim(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figure 8.7:** Simulateur EP08_07: Morphologie avec Connectivité Configurable et Mesure


In [ ]:
%%writefile EP08_07.py
# Code Python

In [ ]:
TestSuite("EP08_07.py").run()

### EP08_08 🟡 Image en Niveaux de Gris et Seuillage Dynamique

Dans cet exercice, l'image d'entrée n'est plus strictement binaire (`0`/`1`) mais devient une **image en niveaux de gris ($8$ bits, $0\dots255$)**, où les objets possèdent une intensité moyenne intermédiaire sur un fond sombre ($0$), avec en plus un bruit de type sel et poivre réparti sur toute l'image.

#### 📋 Directives d'Implémentation

1. **Entrée :** lire $H$ et $W$ sur la première ligne, suivis des $H$ lignes contenant des valeurs entières de $0$ à $255$ dans une matrice $H \times W$.
2. **Prétraitement :**
* Appliquer un filtre **Médian ($3 \times 3$)** pour éliminer le bruit sel et poivre tout en préservant des bords nets.
* Appliquer un **Seuillage d'Otsu** (ou un seuil fixe $T = 60$) pour binariser l'image nettoyée.


3. **Mesure et Sortie :** extraire le contour des objets, calculer les métriques géométriques (`area`, `perimeter`, `cx`, `cy`, `x`, `y`, `w`, `h`, `circularity`, `solidity`) et trier les objets par `bbox[0]` (et `bbox[1]` en cas d'égalité). Réattribuer l'`id` de $1$ à $N$ et imprimer le tableau.
   - Pour le tri, utiliser `medidas.sort(key=lambda m: (m['bbox'][1], m['bbox'][0]))`, avec `medidas = mm.measure(img)`.


#### 📌 Exemples

| Entrée | Sortie |
|---|---|
| 16 32<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>...<br>(image binaire contenant un carré et un cercle) | id area perimeter cx cy x y w h circularity solidity vertices<br>1 16.0 16.0 8.0 8.0 6 6 5 5 0.79 1.000 4<br>2 28.3 18.8 22.5 8.0 19 5 7 7 1.00 1.000 8 |

\newpage

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0808" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧮 Simulateur EP08_08 : Bruit sel et poivre en niveaux de gris & Mesure OpenCV</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">Médiane 3x3 &rarr; Binarisation &rarr; Mesure</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:flex-end;margin-bottom:14px;">
      <div style="flex:2;min-width:260px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">ÉTAPE DU TRAITEMENT</div>
        <div style="display:flex;gap:4px;background:#f1ead7;border:1px solid #e4dcc8;border-radius:11px;padding:3px;">
          <button id="ep0808_stage0" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:700;border:none;background:#26241d;color:#fbf7ee;cursor:pointer;border-radius:9px;white-space:nowrap;">1. Gris bruité</button>
          <button id="ep0808_stage1" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">2. Médiane 3x3</button>
          <button id="ep0808_stage2" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">3. Binarisée (Otsu)</button>
        </div>
      </div>

      <div>
        <button id="ep0808_btnRand" style="background:#26241d;color:#7ee7c6;border:none;padding:8px 14px;font-size:11px;font-weight:700;border-radius:9px;cursor:pointer;display:inline-flex;align-items:center;gap:6px;font-family:monospace;">🎲 Générer des formes/positions aléatoires</button>
      </div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;margin-bottom:14px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:8px;text-align:left;">AFFICHAGE DE LA MATRICE DE PIXELS</div>
      <div id="ep0808_grid_container" style="overflow-x:auto;padding-bottom:4px;"></div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">TABLEAU DES MESURES DES OBJETS (TRIÉS PAR BBOX_X, BBOX_Y)</div>
      <div style="overflow-x:auto;">
        <table style="width:100%;border-collapse:collapse;font-size:10.5px;font-family:monospace;margin-top:10px;">
          <thead>
            <tr style="background:#f1ead7;">
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">id</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">area</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">perimeter</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cx</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cy</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">x</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">y</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">w</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">h</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">circularity</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">solidity</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">vertices</th>
            </tr>
          </thead>
          <tbody id="ep0808_tbody"></tbody>
        </table>
      </div>
    </div>
  </div>
</div>

<script>
(function(){
  function initSim(root){
    if (!root || root.dataset.ep0808Init) return;
    root.dataset.ep0808Init = "1";

    var H = 10, W = 20, stage = 0;
    var matOrig = [], matMed = [], matBin = [], medidasObjetos = [];

    var elBtnRand = root.querySelector('#ep0808_btnRand');
    var elGridContainer = root.querySelector('#ep0808_grid_container');
    var elTbody = root.querySelector('#ep0808_tbody');

    function cv2_findContours(mat) {
      var visited = Array.from({length: H}, function(){ return Array(W).fill(false); });
      var contours = [];
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          if (mat[r][c] === 1 && !visited[r][c]) {
            var blobPixels = [], queue = [[r, c]];
            visited[r][c] = true;
            while (queue.length > 0) {
              var curr = queue.shift(), cr = curr[0], cc = curr[1];
              blobPixels.push([cr, cc]);
              var dirs = [[-1,0],[1,0],[0,-1],[0,1],[-1,-1],[-1,1],[1,-1],[1,1]];
              for (var d = 0; d < dirs.length; d++) {
                var nr = cr + dirs[d][0], nc = cc + dirs[d][1];
                if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
                  if (mat[nr][nc] === 1 && !visited[nr][nc]) {
                    visited[nr][nc] = true; queue.push([nr, nc]);
                  }
                }
              }
            }
            var borderPts = [];
            blobPixels.forEach(function(p) {
              var pr = p[0], pc = p[1];
              if (pr === 0 || pr === H-1 || pc === 0 || pc === W-1 ||
                  mat[pr-1][pc] === 0 || mat[pr+1][pc] === 0 ||
                  mat[pr][pc-1] === 0 || mat[pr][pc+1] === 0) { borderPts.push([pc, pr]); }
            });
            var cx = 0, cy = 0;
            borderPts.forEach(function(pt){ cx += pt[0]; cy += pt[1]; });
            cx /= borderPts.length; cy /= borderPts.length;
            borderPts.sort(function(a, b) { return Math.atan2(a[1] - cy, a[0] - cx) - Math.atan2(b[1] - cy, b[0] - cx); });
            contours.push({ pixels: blobPixels, contour: borderPts });
          }
        }
      }
      return contours;
    }

    function cv2_contourArea(contour) {
      if (contour.length < 3) return 0.0;
      var area = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        area += contour[i][0] * contour[j][1] - contour[j][0] * contour[i][1];
      }
      return Math.abs(area) / 2.0;
    }

    function cv2_arcLength(contour) {
      if (contour.length < 2) return 0.0;
      var per = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        per += Math.hypot(contour[j][0] - contour[i][0], contour[j][1] - contour[i][1]);
      }
      return per;
    }

    function cv2_moments(blobPixels) {
      var m00 = blobPixels.length, m10 = 0.0, m01 = 0.0;
      blobPixels.forEach(function(p) { m10 += p[1]; m01 += p[0]; });
      return { m00: m00, cx: m00 > 0 ? m10 / m00 : 0, cy: m00 > 0 ? m01 / m00 : 0 };
    }

    function cv2_boundingRect(blobPixels) {
      var minX = W, maxX = 0, minY = H, maxY = 0;
      blobPixels.forEach(function(p) {
        var r = p[0], c = p[1];
        if (c < minX) minX = c; if (c > maxX) maxX = c;
        if (r < minY) minY = r; if (r > maxY) maxY = r;
      });
      return { x: minX, y: minY, w: maxX - minX + 1, h: maxY - minY + 1 };
    }

    function cv2_convexHull(points) {
      if (points.length <= 2) return points;
      var pts = points.slice().sort(function(a,b){ return a[0] === b[0] ? a[1] - b[1] : a[0] - b[0]; });
      function cross(o, a, b) { return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0]); }
      var lower = [];
      for (var i = 0; i < pts.length; i++) {
        while (lower.length >= 2 && cross(lower[lower.length - 2], lower[lower.length - 1], pts[i]) <= 0) lower.pop();
        lower.push(pts[i]);
      }
      var upper = [];
      for (var i = pts.length - 1; i >= 0; i--) {
        while (upper.length >= 2 && cross(upper[upper.length - 2], upper[upper.length - 1], pts[i]) <= 0) upper.pop();
        upper.push(pts[i]);
      }
      upper.pop(); lower.pop();
      return lower.concat(upper);
    }

    function cv2_approxPolyDP(contour, precision) {
      if (contour.length <= 2) return contour;
      var epsilon = precision * cv2_arcLength(contour);
      function rdp(pts, eps) {
        if (pts.length <= 2) return pts;
        var dmax = 0, index = 0, end = pts.length - 1;
        for (var i = 1; i < end; i++) {
          var dx = pts[end][0] - pts[0][0], dy = pts[end][1] - pts[0][1];
          var mag = Math.hypot(dx, dy);
          var d = mag === 0 ? Math.hypot(pts[i][0] - pts[0][0], pts[i][1] - pts[0][1]) :
            Math.abs(dy * pts[i][0] - dx * pts[i][1] + pts[end][0] * pts[0][1] - pts[end][1] * pts[0][0]) / mag;
          if (d > dmax) { index = i; dmax = d; }
        }
        if (dmax > eps) {
          var r1 = rdp(pts.slice(0, index + 1), eps);
          var r2 = rdp(pts.slice(index, end + 1), eps);
          return r1.slice(0, r1.length - 1).concat(r2);
        } else { return [pts[0], pts[end]]; }
      }
      return rdp(contour, epsilon);
    }

    function measureOpenCV(mat) {
      var blobs = cv2_findContours(mat);
      var medidas = [];
      blobs.forEach(function(item) {
        var contour = item.contour, pixels = item.pixels;
        var area = cv2_contourArea(contour);
        if (area === 0) area = pixels.length;
        var per = cv2_arcLength(contour);
        var bbox = cv2_boundingRect(pixels);
        var moments = cv2_moments(pixels);
        var hull = cv2_convexHull(contour);
        var hull_area = cv2_contourArea(hull);
        if (hull_area === 0) hull_area = area;
        var poly = cv2_approxPolyDP(contour, 0.02);
        medidas.push({
          area: area, perimeter: per, cx: moments.cx, cy: moments.cy,
          x: bbox.x, y: bbox.y, w: bbox.w, h: bbox.h,
          circularity: per > 0 ? (4 * Math.PI * area) / (per * per) : 0,
          solidity: hull_area > 0 ? area / hull_area : 0, vertices: poly.length
        });
      });
      medidas.sort(function(a, b) { return a.x !== b.x ? a.x - b.x : a.y - b.y; });
      medidas.forEach(function(m, i) { m.id = i + 1; });
      return medidas;
    }

    function gerarCenario() {
      matOrig = Array.from({length: H}, function(){ return Array(W).fill(0); });
      var w1 = Math.floor(Math.random() * 2) + 4, h1 = Math.floor(Math.random() * 2) + 4;
      var x1 = Math.floor(Math.random() * 2) + 1, y1 = Math.floor(Math.random() * 2) + 1;
      var val1 = 150;
      for (var r = y1; r < y1 + h1; r++) { for (var c = x1; c < x1 + w1; c++) matOrig[r][c] = val1; }

      var w2 = Math.floor(Math.random() * 2) + 4, h2 = Math.floor(Math.random() * 2) + 4;
      var x2 = Math.floor(Math.random() * 2) + 11, y2 = Math.floor(Math.random() * 2) + 2;
      var val2 = 180;
      for (var r = y2; r < y2 + h2; r++) { for (var c = x2; c < x2 + w2; c++) matOrig[r][c] = val2; }

      for (var i = 0; i < 4; i++) {
        var sr = Math.floor(Math.random() * H), sc = Math.floor(Math.random() * W);
        if (matOrig[sr][sc] === 0) matOrig[sr][sc] = 255;
      }
      matOrig[y1 + 1][x1 + 1] = 0; matOrig[y2 + 1][x2 + 1] = 0;

      matMed = JSON.parse(JSON.stringify(matOrig));
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          var vals = [];
          for (var dr = -1; dr <= 1; dr++) {
            for (var dc = -1; dc <= 1; dc++) {
              var nr = r + dr, nc = c + dc;
              if (nr >= 0 && nr < H && nc >= 0 && nc < W) { vals.push(matOrig[nr][nc]); } else { vals.push(0); }
            }
          }
          vals.sort(function(a, b){ return a - b; });
          matMed[r][c] = vals[4];
        }
      }

      matBin = matMed.map(function(row) { return row.map(function(v) { return v > 50 ? 1 : 0; }); });
      medidasObjetos = measureOpenCV(matBin);
      renderGrid();
      renderTabela();
    }

    function renderGrid(){
      elGridContainer.innerHTML = '';
      var grid = document.createElement('div');
      grid.style.cssText = 'display:inline-grid;gap:1px;background:#e4dcc8;padding:1px;border-radius:6px;overflow:auto;max-width:100%;grid-template-columns:repeat(' + W + ', 22px);';
      var currentMat = stage === 0 ? matOrig : (stage === 1 ? matMed : matBin);

      for (var r = 0; r < H; r++){
        for (var c = 0; c < W; c++){
          var v = currentMat[r][c], cell = document.createElement('div');
          if (stage === 2) {
            var bgBin = v === 1 ? '#26241d' : '#ffffff', fgBin = v === 1 ? '#7ee7c6' : '#8a8371';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:' + bgBin + ';color:' + fgBin + ';';
          } else {
            var fgCinza = v > 128 ? '#000000' : '#ffffff';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:rgb(' + v + ',' + v + ',' + v + ');color:' + fgCinza + ';';
          }
          cell.textContent = v;
          grid.appendChild(cell);
        }
      }
      elGridContainer.appendChild(grid);
    }

    function renderTabela(){
      elTbody.innerHTML = '';
      if (medidasObjetos.length === 0) {
        elTbody.innerHTML = '<tr><td colspan="12" style="padding:12px;color:#8a8371;text-align:center;">Nenhum objeto detectado após a filtragem.</td></tr>';
        return;
      }
      medidasObjetos.forEach(function(m, i){
        var tr = document.createElement('tr');
        if (i % 2 === 1) tr.style.background = '#fafaf7';
        tr.innerHTML = 
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;"><b>' + m.id + '</b></td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.area.toFixed(1) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.perimeter.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cx.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cy.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.x + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.y + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.w + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.h + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.circularity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.solidity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.vertices + '</td>';
        elTbody.appendChild(tr);
      });
    }

    function setStage(s, btn){
      [root.querySelector('#ep0808_stage0'), root.querySelector('#ep0808_stage1'), root.querySelector('#ep0808_stage2')].forEach(function(b){
        b.style.background = 'transparent'; b.style.color = '#8a8371'; b.style.fontWeight = '600';
      });
      btn.style.background = '#26241d'; btn.style.color = '#fbf7ee'; btn.style.fontWeight = '700';
      stage = s; renderGrid();
    }

    elBtnRand.addEventListener('click', gerarCenario);
    root.querySelector('#ep0808_stage0').addEventListener('click', function(){ setStage(0, this); });
    root.querySelector('#ep0808_stage1').addEventListener('click', function(){ setStage(1, this); });
    root.querySelector('#ep0808_stage2').addEventListener('click', function(){ setStage(2, this); });

    gerarCenario();
  }

  function tryInit(){
    var root = document.getElementById('sim-ep0808');
    if (root) initSim(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figure 8.8:** Simulateur EP08_08 : Filtrage Médian en Niveaux de Gris et Mesure d


In [ ]:
%%writefile EP08_08.py
# Code Python

In [ ]:
TestSuite("EP08_08.py").run()

### EP08_09 🟠 Gradient d’éclairage et seuillage adaptatif

Dans cette variante, les objets sont immergés dans un fond avec un **éclairage non uniforme (gradient doux d’éclairage)**. Le seuillage simple par valeur unique échoue, nécessitant un prétraitement plus robuste.

#### 📋 Directives d’implémentation

1. **Entrée :** image en niveaux de gris $H \times W$ avec une variation de fond de $20$ à $180$.
2. **Prétraitement :**

* Appliquer un **seuillage adaptatif** (ex. : `cv2.adaptiveThreshold` avec une fenêtre gaussienne de $15 \times 15$ et une constante $C = 3$) pour isoler les objets indépendamment de la variation du fond.

  `cv2.adaptiveThreshold(img_gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, ksize, C) // 255`

  `ksize` et `C` sont lus après l’image.

* Opération morphologique de **fermeture** ($3 \times 3$) pour sceller d’éventuelles défaillances dans les contours.

1. **Mesure et classification :** extraire les mesures.

4. **Tri et sortie :** trier par `(bbox[0], bbox[1])` et imprimer le tableau incluant la colonne `class`.
   - Pour trier, utiliser `medidas.sort(key=lambda m: (m['bbox'][1], m['bbox'][0]))`, avec `medidas = mm.measure(img, precision=0.02)`.

#### 📌 Exemples

| Entrée | Sortie |
|---|---|
| 16 32<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>...<br>k 20 | id area perimeter cx cy x y w h circularity solidity vertices<br>1 9.0 12.0 10.0 5.0 8 3 5 5 0.79 1.000 4<br>2 28.3 18.8 25.0 12.0 22 9 7 7 1.00 1.000 3 |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0809" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧮 Simulateur EP08_09 : Gradient d'éclairage et seuil adaptatif & Mesure OpenCV</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">Adaptatif vs Global &rarr; Mesure</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:flex-end;margin-bottom:14px;">
      <div style="flex:2;min-width:280px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">ÉTAPE DU TRAITEMENT</div>
        <div style="display:flex;gap:4px;background:#f1ead7;border:1px solid #e4dcc8;border-radius:11px;padding:3px;">
          <button id="ep0809_stage0" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:700;border:none;background:#26241d;color:#fbf7ee;cursor:pointer;border-radius:9px;white-space:nowrap;">1. Gradient de gris</button>
          <button id="ep0809_stage1" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">2. Seuil global échoué</button>
          <button id="ep0809_stage2" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">3. Seuil adaptatif OK</button>
        </div>
      </div>

      <div>
        <button id="ep0809_btnRand" style="background:#26241d;color:#7ee7c6;border:none;padding:8px 14px;font-size:11px;font-weight:700;border-radius:9px;cursor:pointer;display:inline-flex;align-items:center;gap:6px;font-family:monospace;">🎲 Générer des formes/positions aléatoires</button>
      </div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;margin-bottom:14px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:8px;text-align:left;">VISUALISATION DE LA MATRICE DE PIXELS</div>
      <div id="ep0809_grid_container" style="overflow-x:auto;padding-bottom:4px;"></div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">TABLEAU DES MESURES DES OBJETS (CALCULÉ AU SEUIL ADAPTATIF OK)</div>
      <div style="overflow-x:auto;">
        <table style="width:100%;border-collapse:collapse;font-size:10.5px;font-family:monospace;margin-top:10px;">
          <thead>
            <tr style="background:#f1ead7;">
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">id</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">aire</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">périmètre</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cx</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cy</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">x</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">y</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">w</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">h</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">circularité</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">solidité</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">sommets</th>
            </tr>
          </thead>
          <tbody id="ep0809_tbody"></tbody>
        </table>
      </div>
    </div>
  </div>
</div>

<script>
(function(){
  function initSim(root){
    if (!root || root.dataset.ep0809Init) return;
    root.dataset.ep0809Init = "1";

    var H = 10, W = 20, stage = 0;
    var matGrad = [], matGlob = [], matAdapt = [], medidasObjetos = [];

    var elBtnRand = root.querySelector('#ep0809_btnRand');
    var elGridContainer = root.querySelector('#ep0809_grid_container');
    var elTbody = root.querySelector('#ep0809_tbody');

    function cv2_findContours(mat) {
      var visited = Array.from({length: H}, function(){ return Array(W).fill(false); });
      var contours = [];
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          if (mat[r][c] === 1 && !visited[r][c]) {
            var blobPixels = [], queue = [[r, c]];
            visited[r][c] = true;
            while (queue.length > 0) {
              var curr = queue.shift(), cr = curr[0], cc = curr[1];
              blobPixels.push([cr, cc]);
              var dirs = [[-1,0],[1,0],[0,-1],[0,1],[-1,-1],[-1,1],[1,-1],[1,1]];
              for (var d = 0; d < dirs.length; d++) {
                var nr = cr + dirs[d][0], nc = cc + dirs[d][1];
                if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
                  if (mat[nr][nc] === 1 && !visited[nr][nc]) {
                    visited[nr][nc] = true; queue.push([nr, nc]);
                  }
                }
              }
            }
            var borderPts = [];
            blobPixels.forEach(function(p) {
              var pr = p[0], pc = p[1];
              if (pr === 0 || pr === H-1 || pc === 0 || pc === W-1 ||
                  mat[pr-1][pc] === 0 || mat[pr+1][pc] === 0 ||
                  mat[pr][pc-1] === 0 || mat[pr][pc+1] === 0) { borderPts.push([pc, pr]); }
            });
            var cx = 0, cy = 0;
            borderPts.forEach(function(pt){ cx += pt[0]; cy += pt[1]; });
            cx /= borderPts.length; cy /= borderPts.length;
            borderPts.sort(function(a, b) { return Math.atan2(a[1] - cy, a[0] - cx) - Math.atan2(b[1] - cy, b[0] - cx); });
            contours.push({ pixels: blobPixels, contour: borderPts });
          }
        }
      }
      return contours;
    }

    function cv2_contourArea(contour) {
      if (contour.length < 3) return 0.0;
      var area = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        area += contour[i][0] * contour[j][1] - contour[j][0] * contour[i][1];
      }
      return Math.abs(area) / 2.0;
    }

    function cv2_arcLength(contour) {
      if (contour.length < 2) return 0.0;
      var per = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        per += Math.hypot(contour[j][0] - contour[i][0], contour[j][1] - contour[i][1]);
      }
      return per;
    }

    function cv2_moments(blobPixels) {
      var m00 = blobPixels.length, m10 = 0.0, m01 = 0.0;
      blobPixels.forEach(function(p) { m10 += p[1]; m01 += p[0]; });
      return { m00: m00, cx: m00 > 0 ? m10 / m00 : 0, cy: m00 > 0 ? m01 / m00 : 0 };
    }

    function cv2_boundingRect(blobPixels) {
      var minX = W, maxX = 0, minY = H, maxY = 0;
      blobPixels.forEach(function(p) {
        var r = p[0], c = p[1];
        if (c < minX) minX = c; if (c > maxX) maxX = c;
        if (r < minY) minY = r; if (r > maxY) maxY = r;
      });
      return { x: minX, y: minY, w: maxX - minX + 1, h: maxY - minY + 1 };
    }

    function cv2_convexHull(points) {
      if (points.length <= 2) return points;
      var pts = points.slice().sort(function(a,b){ return a[0] === b[0] ? a[1] - b[1] : a[0] - b[0]; });
      function cross(o, a, b) { return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0]); }
      var lower = [];
      for (var i = 0; i < pts.length; i++) {
        while (lower.length >= 2 && cross(lower[lower.length - 2], lower[lower.length - 1], pts[i]) <= 0) lower.pop();
        lower.push(pts[i]);
      }
      var upper = [];
      for (var i = pts.length - 1; i >= 0; i--) {
        while (upper.length >= 2 && cross(upper[upper.length - 2], upper[upper.length - 1], pts[i]) <= 0) upper.pop();
        upper.push(pts[i]);
      }
      upper.pop(); lower.pop();
      return lower.concat(upper);
    }

    function cv2_approxPolyDP(contour, precision) {
      if (contour.length <= 2) return contour;
      var epsilon = precision * cv2_arcLength(contour);
      function rdp(pts, eps) {
        if (pts.length <= 2) return pts;
        var dmax = 0, index = 0, end = pts.length - 1;
        for (var i = 1; i < end; i++) {
          var dx = pts[end][0] - pts[0][0], dy = pts[end][1] - pts[0][1];
          var mag = Math.hypot(dx, dy);
          var d = mag === 0 ? Math.hypot(pts[i][0] - pts[0][0], pts[i][1] - pts[0][1]) :
            Math.abs(dy * pts[i][0] - dx * pts[i][1] + pts[end][0] * pts[0][1] - pts[end][1] * pts[0][0]) / mag;
          if (d > dmax) { index = i; dmax = d; }
        }
        if (dmax > eps) {
          var r1 = rdp(pts.slice(0, index + 1), eps);
          var r2 = rdp(pts.slice(index, end + 1), eps);
          return r1.slice(0, r1.length - 1).concat(r2);
        } else { return [pts[0], pts[end]]; }
      }
      return rdp(contour, epsilon);
    }

    function measureOpenCV(mat) {
      var blobs = cv2_findContours(mat);
      var medidas = [];
      blobs.forEach(function(item) {
        var contour = item.contour, pixels = item.pixels;
        var area = cv2_contourArea(contour);
        if (area === 0) area = pixels.length;
        var per = cv2_arcLength(contour);
        var bbox = cv2_boundingRect(pixels);
        var moments = cv2_moments(pixels);
        var hull = cv2_convexHull(contour);
        var hull_area = cv2_contourArea(hull);
        if (hull_area === 0) hull_area = area;
        var poly = cv2_approxPolyDP(contour, 0.02);
        medidas.push({
          area: area, perimeter: per, cx: moments.cx, cy: moments.cy,
          x: bbox.x, y: bbox.y, w: bbox.w, h: bbox.h,
          circularity: per > 0 ? (4 * Math.PI * area) / (per * per) : 0,
          solidity: hull_area > 0 ? area / hull_area : 0, vertices: poly.length
        });
      });
      medidas.sort(function(a, b) { return a.x !== b.x ? a.x - b.x : a.y - b.y; });
      medidas.forEach(function(m, i) { m.id = i + 1; });
      return medidas;
    }

    function gerarCenario() {
      matGrad = Array.from({length: H}, function(_, r){
        return Array.from({length: W}, function(_, c){ return Math.round(20 + c * 9.5); });
      });
      var w1 = 3, h1 = 3;
      var x1 = Math.floor(Math.random() * 2) + 2, y1 = Math.floor(Math.random() * 2) + 2;
      for (var r = y1; r < y1 + h1; r++) { for (var c = x1; c < x1 + w1; c++) matGrad[r][c] += 90; }

      var w2 = 3, h2 = 3;
      var x2 = Math.floor(Math.random() * 2) + 14, y2 = Math.floor(Math.random() * 2) + 2;
      for (var r = y2; r < y2 + h2; r++) { for (var c = x2; c < x2 + w2; c++) matGrad[r][c] += 90; }

      matGlob = matGrad.map(function(row) { return row.map(function(v) { return v > 110 ? 1 : 0; }); });

      var blockSize = 5, half = Math.floor(blockSize / 2), C_val = 20;
      matAdapt = Array.from({length: H}, function(){ return Array(W).fill(0); });
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          var sum = 0, count = 0;
          for (var dr = -half; dr <= half; dr++) {
            for (var dc = -half; dc <= half; dc++) {
              var nr = r + dr, nc = c + dc;
              if (nr >= 0 && nr < H && nc >= 0 && nc < W) { sum += matGrad[nr][nc]; count++; }
            }
          }
          var mean = sum / count;
          matAdapt[r][c] = matGrad[r][c] > (mean + C_val) ? 1 : 0;
        }
      }
      medidasObjetos = measureOpenCV(matAdapt);
      renderGrid();
      renderTabela();
    }

    function renderGrid(){
      elGridContainer.innerHTML = '';
      var grid = document.createElement('div');
      grid.style.cssText = 'display:inline-grid;gap:1px;background:#e4dcc8;padding:1px;border-radius:6px;overflow:auto;max-width:100%;grid-template-columns:repeat(' + W + ', 22px);';
      var currentMat = stage === 0 ? matGrad : (stage === 1 ? matGlob : matAdapt);

      for (var r = 0; r < H; r++){
        for (var c = 0; c < W; c++){
          var v = currentMat[r][c], cell = document.createElement('div');
          if (stage > 0) {
            var bgBin = v === 1 ? '#26241d' : '#ffffff', fgBin = v === 1 ? '#7ee7c6' : '#8a8371';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:' + bgBin + ';color:' + fgBin + ';';
          } else {
            var fgCinza = v > 120 ? '#ffffff' : '#374151';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:rgb(' + (255 - v) + ',' + (255 - v) + ',' + (255 - v) + ');color:' + fgCinza + ';';
          }
          cell.textContent = v;
          grid.appendChild(cell);
        }
      }
      elGridContainer.appendChild(grid);
    }

    function renderTabela(){
      elTbody.innerHTML = '';
      if (medidasObjetos.length === 0) {
        elTbody.innerHTML = '<tr><td colspan="12" style="padding:12px;color:#8a8371;text-align:center;">Nenhum objeto detectado após o limiar adaptativo.</td></tr>';
        return;
      }
      medidasObjetos.forEach(function(m, i){
        var tr = document.createElement('tr');
        if (i % 2 === 1) tr.style.background = '#fafaf7';
        tr.innerHTML = 
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;"><b>' + m.id + '</b></td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.area.toFixed(1) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.perimeter.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cx.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cy.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.x + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.y + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.w + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.h + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.circularity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.solidity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.vertices + '</td>';
        elTbody.appendChild(tr);
      });
    }

    function setStage(s, btn){
      [root.querySelector('#ep0809_stage0'), root.querySelector('#ep0809_stage1'), root.querySelector('#ep0809_stage2')].forEach(function(b){
        b.style.background = 'transparent'; b.style.color = '#8a8371'; b.style.fontWeight = '600';
      });
      btn.style.background = '#26241d'; btn.style.color = '#fbf7ee'; btn.style.fontWeight = '700';
      stage = s; renderGrid();
    }

    elBtnRand.addEventListener('click', gerarCenario);
    root.querySelector('#ep0809_stage0').addEventListener('click', function(){ setStage(0, this); });
    root.querySelector('#ep0809_stage1').addEventListener('click', function(){ setStage(1, this); });
    root.querySelector('#ep0809_stage2').addEventListener('click', function(){ setStage(2, this); });

    gerarCenario();
  }

  function tryInit(){
    var root = document.getElementById('sim-ep0809');
    if (root) initSim(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figure 8.9:** Simulateur EP08_09 : Gradient d


In [ ]:
%%writefile EP08_09.py
# Code Python

In [ ]:
TestSuite("EP08_09.py").run()

### EP08_10 🔴 Contraste Faible et Séparation d'Objets Thèmes (*Watershed* / Distance)

Dans cet exercice, **certains objets géométriques sont légèrement en contact (superposés sur les bords)**. La simple extraction des contours traiterait deux objets comme s'ils n'en formaient qu'un seul.

#### 📋 Directives d'Implémentation

1. **Entrée :** matrice $H \times W$ en niveaux de gris avec des objets d'intensité $110\dots140$ sur un fond $0$, avec bruit et une paire d'objets tangents.
2. **Prétraitement et Séparation :**
* Application du seuillage.
* Application de la **Transformée de Distance** (`mm.dist`).
* Obtention des pics de distance pour servir de marqueurs dans la **Transformée *Watershed*** (`mm.watershed`), séparant physiquement les objets en contact dans le masque. **Astuce :** utiliser `mm.regmax()` pour obtenir les maxima locaux, puis les étiqueter avec `mm.label0`.
* Après le *watershed*, appliquer à nouveau le seuillage avec `mm.threshold(water,0)//255`.

3. **Analyse des Composants Connectés :** mesurer chaque région isolée après le *Watershed*.
4. **Sortie :** imprimer les composants triés par `(bbox[0], bbox[1])` avec leurs métriques individuelles de surface, centroïde et solidité.
   - Pour trier, utiliser `medidas.sort(key=lambda m: (m['bbox'][1], m['bbox'][0]))`, avec `medidas = mm.measure(img, precision=0.02)`.



#### 📌 Exemples

| Entrée | Sortie |
|---|---|
| 16 16<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>...<br>(image binaire contenant deux carrés) | id area perimeter cx cy x y w h solidity<br>1 16.0 16.0 5.0 5.0 3 3 5 5 1.000<br>2 16.0 16.0 11.0 5.0 9 3 5 5 1.000 |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0810" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧮 Simulateur EP08_10 : Séparation de Disques Tangents (Transformée L2 & Watershed)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">mm.dist L2 &rarr; mm.watershed &rarr; Mesure</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:flex-end;margin-bottom:14px;">
      <div style="flex:2;min-width:300px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">ÉTAPE DU TRAITEMENT MORPHOLOGIQUE</div>
        <div style="display:flex;gap:4px;background:#f1ead7;border:1px solid #e4dcc8;border-radius:11px;padding:3px;">
          <button id="ep0810_stage0" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:700;border:none;background:#26241d;color:#fbf7ee;cursor:pointer;border-radius:9px;white-space:nowrap;">1. Gris bruité</button>
          <button id="ep0810_stage1" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">2. Masque uni</button>
          <button id="ep0810_stage2" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">3. Distance L2</button>
          <button id="ep0810_stage3" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">4. Watershed (Coupe)</button>
        </div>
      </div>

      <div>
        <button id="ep0810_btnRand" style="background:#26241d;color:#7ee7c6;border:none;padding:8px 14px;font-size:11px;font-weight:700;border-radius:9px;cursor:pointer;display:inline-flex;align-items:center;gap:6px;font-family:monospace;">🎲 Générer des Disques avec Rayons Aléatoires</button>
      </div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;margin-bottom:14px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:8px;text-align:left;">VISUALISATION DE LA MATRICE DE PIXELS</div>
      <div id="ep0810_grid_container" style="overflow-x:auto;padding-bottom:4px;"></div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">TABLEAU DES MESURES DES DISQUES APRÈS LA COUPE DU WATERSHED</div>
      <div style="overflow-x:auto;">
        <table style="width:100%;border-collapse:collapse;font-size:10.5px;font-family:monospace;margin-top:10px;">
          <thead>
            <tr style="background:#f1ead7;">
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">id</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">aire</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">périmètre</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cx</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cy</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">x</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">y</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">w</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">h</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">circularité</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">solidité</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">sommets</th>
            </tr>
          </thead>
          <tbody id="ep0810_tbody"></tbody>
        </table>
      </div>
    </div>
  </div>
</div>

<script>
(function(){
  function initSim(root){
    if (!root || root.dataset.ep0810Init) return;
    root.dataset.ep0810Init = "1";

    var H = 11, W = 21, stage = 0;
    var matOrig = [], matBin = [], matDist = [], matWash = [], medidasObjetos = [];

    var elBtnRand = root.querySelector('#ep0810_btnRand');
    var elGridContainer = root.querySelector('#ep0810_grid_container');
    var elTbody = root.querySelector('#ep0810_tbody');

    function cv2_findContours(mat) {
      var visited = Array.from({length: H}, function(){ return Array(W).fill(false); });
      var contours = [];
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          if (mat[r][c] === 1 && !visited[r][c]) {
            var blobPixels = [], queue = [[r, c]];
            visited[r][c] = true;
            while (queue.length > 0) {
              var curr = queue.shift(), cr = curr[0], cc = curr[1];
              blobPixels.push([cr, cc]);
              var dirs = [[-1,0],[1,0],[0,-1],[0,1],[-1,-1],[-1,1],[1,-1],[1,1]];
              for (var d = 0; d < dirs.length; d++) {
                var nr = cr + dirs[d][0], nc = cc + dirs[d][1];
                if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
                  if (mat[nr][nc] === 1 && !visited[nr][nc]) {
                    visited[nr][nc] = true; queue.push([nr, nc]);
                  }
                }
              }
            }
            var borderPts = [];
            blobPixels.forEach(function(p) {
              var pr = p[0], pc = p[1];
              if (pr === 0 || pr === H-1 || pc === 0 || pc === W-1 ||
                  mat[pr-1][pc] === 0 || mat[pr+1][pc] === 0 ||
                  mat[pr][pc-1] === 0 || mat[pr][pc+1] === 0) { borderPts.push([pc, pr]); }
            });
            var cx = 0, cy = 0;
            borderPts.forEach(function(pt){ cx += pt[0]; cy += pt[1]; });
            cx /= borderPts.length; cy /= borderPts.length;
            borderPts.sort(function(a, b) { return Math.atan2(a[1] - cy, a[0] - cx) - Math.atan2(b[1] - cy, b[0] - cx); });
            contours.push({ pixels: blobPixels, contour: borderPts });
          }
        }
      }
      return contours;
    }

    function cv2_contourArea(contour) {
      if (contour.length < 3) return 0.0;
      var area = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        area += contour[i][0] * contour[j][1] - contour[j][0] * contour[i][1];
      }
      return Math.abs(area) / 2.0;
    }

    function cv2_arcLength(contour) {
      if (contour.length < 2) return 0.0;
      var per = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        per += Math.hypot(contour[j][0] - contour[i][0], contour[j][1] - contour[i][1]);
      }
      return per;
    }

    function cv2_moments(blobPixels) {
      var m00 = blobPixels.length, m10 = 0.0, m01 = 0.0;
      blobPixels.forEach(function(p) { m10 += p[1]; m01 += p[0]; });
      return { m00: m00, cx: m00 > 0 ? m10 / m00 : 0, cy: m00 > 0 ? m01 / m00 : 0 };
    }

    function cv2_boundingRect(blobPixels) {
      var minX = W, maxX = 0, minY = H, maxY = 0;
      blobPixels.forEach(function(p) {
        var r = p[0], c = p[1];
        if (c < minX) minX = c; if (c > maxX) maxX = c;
        if (r < minY) minY = r; if (r > maxY) maxY = r;
      });
      return { x: minX, y: minY, w: maxX - minX + 1, h: maxY - minY + 1 };
    }

    function cv2_convexHull(points) {
      if (points.length <= 2) return points;
      var pts = points.slice().sort(function(a,b){ return a[0] === b[0] ? a[1] - b[1] : a[0] - b[0]; });
      function cross(o, a, b) { return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0]); }
      var lower = [];
      for (var i = 0; i < pts.length; i++) {
        while (lower.length >= 2 && cross(lower[lower.length - 2], lower[lower.length - 1], pts[i]) <= 0) lower.pop();
        lower.push(pts[i]);
      }
      var upper = [];
      for (var i = pts.length - 1; i >= 0; i--) {
        while (upper.length >= 2 && cross(upper[upper.length - 2], upper[upper.length - 1], pts[i]) <= 0) upper.pop();
        upper.push(pts[i]);
      }
      upper.pop(); lower.pop();
      return lower.concat(upper);
    }

    function cv2_approxPolyDP(contour, precision) {
      if (contour.length <= 2) return contour;
      var epsilon = precision * cv2_arcLength(contour);
      function rdp(pts, eps) {
        if (pts.length <= 2) return pts;
        var dmax = 0, index = 0, end = pts.length - 1;
        for (var i = 1; i < end; i++) {
          var dx = pts[end][0] - pts[0][0], dy = pts[end][1] - pts[0][1];
          var mag = Math.hypot(dx, dy);
          var d = mag === 0 ? Math.hypot(pts[i][0] - pts[0][0], pts[i][1] - pts[0][1]) :
            Math.abs(dy * pts[i][0] - dx * pts[i][1] + pts[end][0] * pts[0][1] - pts[end][1] * pts[0][0]) / mag;
          if (d > dmax) { index = i; dmax = d; }
        }
        if (dmax > eps) {
          var r1 = rdp(pts.slice(0, index + 1), eps);
          var r2 = rdp(pts.slice(index, end + 1), eps);
          return r1.slice(0, r1.length - 1).concat(r2);
        } else { return [pts[0], pts[end]]; }
      }
      return rdp(contour, epsilon);
    }

    function measureOpenCV(mat) {
      var blobs = cv2_findContours(mat);
      var medidas = [];
      blobs.forEach(function(item) {
        var contour = item.contour, pixels = item.pixels;
        var area = cv2_contourArea(contour);
        if (area === 0) area = pixels.length;
        var per = cv2_arcLength(contour);
        var bbox = cv2_boundingRect(pixels);
        var moments = cv2_moments(pixels);
        var hull = cv2_convexHull(contour);
        var hull_area = cv2_contourArea(hull);
        if (hull_area === 0) hull_area = area;
        var poly = cv2_approxPolyDP(contour, 0.02);
        medidas.push({
          area: area, perimeter: per, cx: moments.cx, cy: moments.cy,
          x: bbox.x, y: bbox.y, w: bbox.w, h: bbox.h,
          circularity: per > 0 ? (4 * Math.PI * area) / (per * per) : 0,
          solidity: hull_area > 0 ? area / hull_area : 0, vertices: poly.length
        });
      });
      medidas.sort(function(a, b) { return a.x !== b.x ? a.x - b.x : a.y - b.y; });
      medidas.forEach(function(m, i) { m.id = i + 1; });
      return medidas;
    }

    function gerarCenario() {
      matOrig = Array.from({length: H}, function(){ return Array(W).fill(0); });
      var r1 = Math.floor(Math.random() * 3) + 2, r2 = Math.floor(Math.random() * 3) + 2; 
      var cy1 = Math.floor(Math.random() * 2) + 4, cx1 = Math.floor(Math.random() * 2) + 3;
      var cx2 = cx1 + r1 + r2, cy2 = cy1;

      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          var d1 = Math.hypot(r - cy1, c - cx1), d2 = Math.hypot(r - cy2, c - cx2);
          if (d1 <= r1 || d2 <= r2) { matOrig[r][c] = 140 + Math.floor(Math.random() * 20); }
        }
      }

      matBin = matOrig.map(function(row) { return row.map(function(v) { return v > 50 ? 1 : 0; }); });
      matDist = Array.from({length: H}, function(){ return Array(W).fill(0); });
      var fundoPixels = [];
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) { if (matBin[r][c] === 0) fundoPixels.push([r, c]); }
      }

      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          if (matBin[r][c] === 1) {
            var minDist = Infinity;
            for (var k = 0; k < fundoPixels.length; k++) {
              var dist = Math.hypot(r - fundoPixels[k][0], c - fundoPixels[k][1]);
              if (dist < minDist) minDist = dist;
            }
            matDist[r][c] = Math.round(minDist);
          }
        }
      }

      matWash = JSON.parse(JSON.stringify(matBin));
      var colCorte = cx1 + r1; 
      for (var r = 0; r < H; r++) { if (matWash[r][colCorte] === 1) matWash[r][colCorte] = 0; }

      medidasObjetos = measureOpenCV(matWash);
      renderGrid();
      renderTabela();
    }

    function renderGrid(){
      elGridContainer.innerHTML = '';
      var grid = document.createElement('div');
      grid.style.cssText = 'display:inline-grid;gap:1px;background:#e4dcc8;padding:1px;border-radius:6px;overflow:auto;max-width:100%;grid-template-columns:repeat(' + W + ', 22px);';

      for (var r = 0; r < H; r++){
        for (var c = 0; c < W; c++){
          var cell = document.createElement('div');
          if (stage === 0) {
            var v = matOrig[r][c]; cell.textContent = v;
            var fgCinza = v > 120 ? '#ffffff' : '#374151';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:rgb(' + (255 - v) + ',' + (255 - v) + ',' + (255 - v) + ');color:' + fgCinza + ';';
          } else if (stage === 1) {
            var v = matBin[r][c]; cell.textContent = v;
            var bgBin = v === 1 ? '#26241d' : '#ffffff', fgBin = v === 1 ? '#7ee7c6' : '#8a8371';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:' + bgBin + ';color:' + fgBin + ';';
          } else if (stage === 2) {
            var v = matDist[r][c]; cell.textContent = v;
            var bgDist = v > 0 ? 'rgb(' + (240 - v * 45) + ',' + (240 - v * 30) + ',255)' : '#ffffff';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:' + bgDist + ';color:#26241d;';
          } else {
            var v = matWash[r][c]; cell.textContent = v;
            var bgWash = v === 1 ? '#26241d' : '#ffffff', fgWash = v === 1 ? '#7ee7c6' : '#8a8371';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:' + bgWash + ';color:' + fgWash + ';';
          }
          grid.appendChild(cell);
        }
      }
      elGridContainer.appendChild(grid);
    }

    function renderTabela(){
      elTbody.innerHTML = '';
      if (medidasObjetos.length === 0) {
        elTbody.innerHTML = '<tr><td colspan="12" style="padding:12px;color:#8a8371;text-align:center;">Nenhum objeto detectado após o corte do Watershed.</td></tr>';
        return;
      }
      medidasObjetos.forEach(function(m, i){
        var tr = document.createElement('tr');
        if (i % 2 === 1) tr.style.background = '#fafaf7';
        tr.innerHTML = 
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;"><b>' + m.id + '</b></td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.area.toFixed(1) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.perimeter.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cx.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cy.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.x + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.y + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.w + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.h + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.circularity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.solidity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.vertices + '</td>';
        elTbody.appendChild(tr);
      });
    }

    function setStage(s, btn){
      [root.querySelector('#ep0810_stage0'), root.querySelector('#ep0810_stage1'), root.querySelector('#ep0810_stage2'), root.querySelector('#ep0810_stage3')].forEach(function(b){
        b.style.background = 'transparent'; b.style.color = '#8a8371'; b.style.fontWeight = '600';
      });
      btn.style.background = '#26241d'; btn.style.color = '#fbf7ee'; btn.style.fontWeight = '700';
      stage = s; renderGrid();
    }

    elBtnRand.addEventListener('click', gerarCenario);
    root.querySelector('#ep0810_stage0').addEventListener('click', function(){ setStage(0, this); });
    root.querySelector('#ep0810_stage1').addEventListener('click', function(){ setStage(1, this); });
    root.querySelector('#ep0810_stage2').addEventListener('click', function(){ setStage(2, this); });
    root.querySelector('#ep0810_stage3').addEventListener('click', function(){ setStage(3, this); });

    gerarCenario();
  }

  function tryInit(){
    var root = document.getElementById('sim-ep0810');
    if (root) initSim(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figure 8.10:** Simulateur EP08_10: Séparation de Disques Tangents via Transformée de Distance L2 et *Watershed*


\newpage

In [ ]:
%%writefile EP08_10.py
# Code Python

In [ ]:
TestSuite("EP08_10.py").run()

### EP08_11 🔴 Classification et Validation d'Objets avec Gabarit de *Bounding Box*

Dans cet exercice, l'objectif est de traiter une image en niveaux de gris contenant plusieurs objets géométriques, d'extraire leurs propriétés avec `mm.measure` et de valider les boîtes englobantes (*bounding boxes*) détectées par rapport à un gabarit réel (*Ground Truth* - GT) fourni en entrée, en utilisant la métrique IoU (*Intersection over Union*).

#### 📋 Directives d'implémentation

1. **Lecture de l'image :** Lire les dimensions $H \times W$ et la matrice $H \times W$ de pixels de l'image en niveaux de gris.
2. ***Pipeline* morphologique :** Binariser l'image via la méthode d'Otsu (`mm.threshold`) et afficher le masque binarisé résultant à l'aide de `mm.drawImg`.
3. **Lecture du gabarit réel (*Ground Truth*) :**

* Lire la quantité $G$ de boîtes englobantes du gabarit.
* Si $G > 0$, lire $G$ lignes contenant chacune 5 valeurs : `id xmin_norm ymin_norm xmax_norm ymax_norm`.
* **Conversion des coordonnées :** Les coordonnées du gabarit sont normalisées dans la plage $[0.0, 1.0]$. Pour les convertir en pixels dans la grille de l'image :

$$x_{\min} = \lfloor \text{xmin\_norm} \times W \rfloor, \quad y_{\min} = \lfloor \text{ymin\_norm} \times H \rfloor$$

$$w = \lfloor \text{xmax\_norm} \times W \rfloor - x_{\min}, \quad h = \lfloor \text{ymax\_norm} \times H \rfloor - y_{\min}$$

4. **Extraction des métriques et calcul de l'IoU :**
* Extraire les propriétés des instances avec `mm.measure(img_bin, precision=0.02)`.
* Pour chaque *bounding box* détectée $(x, y, w, h)$, calculer le chevauchement IoU par rapport aux boîtes du gabarit et définir `hits = 1` s'il existe une correspondance (*match*) avec $\text{IoU} \ge 0.50$, ou `hits = 0` dans le cas contraire.

5. **Sortie :** Trier les instances par position `(bbox[0], bbox[1])` et imprimer le tableau CSV avec la colonne supplémentaire `hits`.
   - Pour le tri, utiliser `medidas.sort(key=lambda m: (m['bbox'][1], m['bbox'][0]))`, avec `medidas = mm.measure(img)`.

#### 🧠 Fondements théoriques et conversion

| Concept | Formule / Opération | Description |
| --- | --- | --- |
| **BBox détectée** | $(x, y, w, h)$ via `mm.measure` | Boîte englobante calculée sur la grille discrète en pixels entiers. |
| **BBox gabarit (GT)** | $(x_{\min}, y_{\min}, w, h)$ convertis | Boîte réelle fournie en entrée en coordonnées relatives $[0.0, 1.0]$. |
| **IoU (Intersection over Union)** | $\text{IoU} = \frac{\text{Aire}(B_{\text{DET}} \cap B_{\text{GT}})}{\text{Aire}(B_{\text{DET}} \cup B_{\text{GT}})}$ | Évalue le taux de chevauchement des boîtes. Est considérée comme valide si $\text{IoU} \ge 0.50$. |
| **Statut de validation (`hits`)** | $1$ si $\max(\text{IoU}) \ge 0.50$, sinon $0$ | Indicateur binaire de succès du détecteur par rapport au gabarit. |

#### 📦 Spécification d'entrée et de sortie (VPL)

**Entrée :**

* **Ligne 1 :** Entiers $H$ et $W$ (dimensions de la matrice).
* **Les $H$ lignes suivantes :** $W$ entiers ($0$ à $255$) représentant l'image en niveaux de gris.
* **Ligne $H + 2$ :** Entier $G$ (quantité de boîtes du gabarit véritable).
* **Les $G$ lignes suivantes :** 5 valeurs numériques par ligne : `id xmin_norm ymin_norm xmax_norm ymax_norm` (où les coordonnées sont des valeurs flottantes entre $0.0$ et $1.0$).

**Sortie :**

1. Matrice binarisée affichée via `mm.drawImg(img_bin)`.
2. En-tête CSV : `id,area,perimeter,cx,cy,x,y,w,h,circularity,solidity,vertices,hits`
3. Une ligne CSV par objet détecté contenant ses propriétés formatées et l'indicateur `hits` ($1$ ou $0$).

#### 📌 Exemples

| Entrée | Sortie |
|---|---|
| 10 20<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 180 0 0 0 0 0 0 0 0 180 180 180 0 0 0 0 0<br>0 0 180 180 180 0 0 0 0 0 0 0 180 180 180 0 0 0 0 0<br>0 0 0 180 0 0 0 0 0 0 0 0 180 180 180 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>2<br>1 0.10 0.30 0.25 0.60<br>2 0.60 0.30 0.75 0.60 | 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 1 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0<br>0 0 1 1 1 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0<br>0 0 0 1 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>id area perimeter cx cy x y w h circularity solidity vertices hits<br>1 2.0 5.7 3.0 4.0 2 3 3 3 0.79 1.000 4 1<br>2 4.0 8.0 13.0 4.0 12 3 3 3 0.79 1.000 4 1 |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0811" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧮 Simulateur EP08_11 : Boîtes englobantes et comparaison IoU avec contrôles indépendants</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">Validation BBox GT vs DET</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:flex-end;margin-bottom:14px;">
      <div style="flex:1.5;min-width:220px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">MODE D'AFFICHAGE</div>
        <div style="display:flex;gap:4px;background:#f1ead7;border:1px solid #e4dcc8;border-radius:11px;padding:3px;">
          <button id="ep0811_stage0" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:700;border:none;background:#26241d;color:#fbf7ee;cursor:pointer;border-radius:9px;white-space:nowrap;">1. Gris original</button>
          <button id="ep0811_stage1" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">2. Binarisée + superpositions</button>
        </div>
      </div>

      <div style="flex:1.5;min-width:240px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">AFFICHAGE DES BOÎTES ENGL OBANTES</div>
        <div style="display:flex;gap:6px;">
          <button id="ep0811_toggleGT" style="padding:5px 10px;font-size:10.5px;font-weight:700;border:1px solid #1d4ed8;background:#2563eb;color:#ffffff;cursor:pointer;border-radius:8px;white-space:nowrap;display:inline-flex;align-items:center;gap:5px;"><span>🟦</span> BBox de référence (GT)</button>
          <button id="ep0811_toggleDET" style="padding:5px 10px;font-size:10.5px;font-weight:700;border:1px solid #047857;background:#059669;color:#ffffff;cursor:pointer;border-radius:8px;white-space:nowrap;display:inline-flex;align-items:center;gap:5px;"><span>🟩</span> BBox détectée (DET)</button>
        </div>
      </div>

      <div>
        <button id="ep0811_btnRand" style="background:#26241d;color:#7ee7c6;border:none;padding:8px 14px;font-size:11px;font-weight:700;border-radius:9px;cursor:pointer;display:inline-flex;align-items:center;gap:6px;font-family:monospace;">🎲 Générer des scènes aléatoires</button>
      </div>
    </div>

    <div style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:10px;padding:8px 12px;margin-bottom:12px;display:flex;gap:16px;flex-wrap:wrap;align-items:center;justify-content:center;">
      <span style="font-size:9.5px;font-weight:700;color:#8a8371;margin-right:4px;">LÉGENDE DES BBOX :</span>
      <div style="display:inline-flex;align-items:center;gap:5px;font-size:10.5px;font-weight:600;color:#374151;"><span style="width:12px;height:12px;border-radius:3px;display:inline-block;background:#2563eb;border:1px dashed #93c5fd;"></span> <span>Référence réelle (GT)</span></div>
      <div style="display:inline-flex;align-items:center;gap:5px;font-size:10.5px;font-weight:600;color:#374151;"><span style="width:12px;height:12px;border-radius:3px;display:inline-block;background:#059669;border:1px solid #34d399;"></span> <span>Détection acceptée (IoU &ge; 0.5)</span></div>
      <div style="display:inline-flex;align-items:center;gap:5px;font-size:10.5px;font-weight:600;color:#374151;"><span style="width:12px;height:12px;border-radius:3px;display:inline-block;background:#dc2626;border:1px solid #f87171;"></span> <span>Détection rejetée (IoU &lt; 0.5)</span></div>
      <div style="display:inline-flex;align-items:center;gap:5px;font-size:10.5px;font-weight:600;color:#374151;"><span style="width:12px;height:12px;border-radius:3px;display:inline-block;background:#7c3aed;border:1px double #a78bfa;"></span> <span>Superposition des BBox</span></div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;margin-bottom:14px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:8px;text-align:left;">VISUALISATION DE LA MATRICE DE PIXELS</div>
      <div id="ep0811_grid_container" style="overflow-x:auto;padding-bottom:4px;"></div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">MESURES, CLASSIFICATION GÉOMÉTRIQUE ET COMPARAISON IOU AVEC LA RÉFÉRENCE</div>
      <div style="overflow-x:auto;">
        <table style="width:100%;border-collapse:collapse;font-size:10.5px;font-family:monospace;margin-top:10px;">
          <thead>
            <tr style="background:#f1ead7;">
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">id</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">classe</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">aire</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">solidité</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">sommets</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">bbox dét (x,y,w,h)</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">bbox gt (x,y,w,h)</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">IoU</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">statut (IoU &ge; 0.5)</th>
            </tr>
          </thead>
          <tbody id="ep0811_tbody"></tbody>
        </table>
      </div>
    </div>
  </div>
</div>

<script>
(function(){
  function initSim(root){
    if (!root || root.dataset.ep0811Init) return;
    root.dataset.ep0811Init = "1";

    var H = 10, W = 20, stage = 0;
    var showGT = true, showDET = true;
    // matOrig/matBin: cena. medidasObjetos: 1 registro por objeto real (GT), casado com sua melhor DET.
    // detBoxesAtuais: caixas "detectadas" simuladas (com ruído/deslocamento em relação ao objeto real).
    var matOrig = [], matBin = [], medidasObjetos = [], detBoxesAtuais = [];

    var elBtnRand = root.querySelector('#ep0811_btnRand');
    var elToggleGT = root.querySelector('#ep0811_toggleGT');
    var elToggleDET = root.querySelector('#ep0811_toggleDET');
    var elGridContainer = root.querySelector('#ep0811_grid_container');
    var elTbody = root.querySelector('#ep0811_tbody');

    function cv2_findContours(mat) {
      var visited = Array.from({length: H}, function(){ return Array(W).fill(false); });
      var contours = [];
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          if (mat[r][c] === 1 && !visited[r][c]) {
            var blobPixels = [], queue = [[r, c]];
            visited[r][c] = true;
            while (queue.length > 0) {
              var curr = queue.shift(), cr = curr[0], cc = curr[1];
              blobPixels.push([cr, cc]);
              var dirs = [[-1,0],[1,0],[0,-1],[0,1],[-1,-1],[-1,1],[1,-1],[1,1]];
              for (var d = 0; d < dirs.length; d++) {
                var nr = cr + dirs[d][0], nc = cc + dirs[d][1];
                if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
                  if (mat[nr][nc] === 1 && !visited[nr][nc]) {
                    visited[nr][nc] = true; queue.push([nr, nc]);
                  }
                }
              }
            }
            var borderPts = [];
            blobPixels.forEach(function(p) {
              var pr = p[0], pc = p[1];
              if (pr === 0 || pr === H-1 || pc === 0 || pc === W-1 ||
                  mat[pr-1][pc] === 0 || mat[pr+1][pc] === 0 ||
                  mat[pr][pc-1] === 0 || mat[pr][pc+1] === 0) { borderPts.push([pc, pr]); }
            });
            var cx = 0, cy = 0;
            borderPts.forEach(function(pt){ cx += pt[0]; cy += pt[1]; });
            cx /= borderPts.length; cy /= borderPts.length;
            borderPts.sort(function(a, b) { return Math.atan2(a[1] - cy, a[0] - cx) - Math.atan2(b[1] - cy, b[0] - cx); });
            contours.push({ pixels: blobPixels, contour: borderPts });
          }
        }
      }
      return contours;
    }

    function cv2_contourArea(contour) {
      if (contour.length < 3) return 0.0;
      var area = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        area += contour[i][0] * contour[j][1] - contour[j][0] * contour[i][1];
      }
      return Math.abs(area) / 2.0;
    }

    function cv2_arcLength(contour) {
      if (contour.length < 2) return 0.0;
      var per = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        per += Math.hypot(contour[j][0] - contour[i][0], contour[j][1] - contour[i][1]);
      }
      return per;
    }

    function cv2_boundingRect(blobPixels) {
      var minX = W, maxX = 0, minY = H, maxY = 0;
      blobPixels.forEach(function(p) {
        var r = p[0], c = p[1];
        if (c < minX) minX = c; if (c > maxX) maxX = c;
        if (r < minY) minY = r; if (r > maxY) maxY = r;
      });
      return { x: minX, y: minY, w: maxX - minX + 1, h: maxY - minY + 1 };
    }

    function cv2_convexHull(points) {
      if (points.length <= 2) return points;
      var pts = points.slice().sort(function(a,b){ return a[0] === b[0] ? a[1] - b[1] : a[0] - b[0]; });
      function cross(o, a, b) { return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0]); }
      var lower = [];
      for (var i = 0; i < pts.length; i++) {
        while (lower.length >= 2 && cross(lower[lower.length - 2], lower[lower.length - 1], pts[i]) <= 0) lower.pop();
        lower.push(pts[i]);
      }
      var upper = [];
      for (var i = pts.length - 1; i >= 0; i--) {
        while (upper.length >= 2 && cross(upper[upper.length - 2], upper[upper.length - 1], pts[i]) <= 0) upper.pop();
        upper.push(pts[i]);
      }
      upper.pop(); lower.pop();
      return lower.concat(upper);
    }

    function cv2_approxPolyDP(contour, precision) {
      if (contour.length <= 2) return contour;
      var epsilon = precision * cv2_arcLength(contour);
      function rdp(pts, eps) {
        if (pts.length <= 2) return pts;
        var dmax = 0, index = 0, end = pts.length - 1;
        for (var i = 1; i < end; i++) {
          var dx = pts[end][0] - pts[0][0], dy = pts[end][1] - pts[0][1];
          var mag = Math.hypot(dx, dy);
          var d = mag === 0 ? Math.hypot(pts[i][0] - pts[0][0], pts[i][1] - pts[0][1]) :
            Math.abs(dy * pts[i][0] - dx * pts[i][1] + pts[end][0] * pts[0][1] - pts[end][1] * pts[0][0]) / mag;
          if (d > dmax) { index = i; dmax = d; }
        }
        if (dmax > eps) {
          var r1 = rdp(pts.slice(0, index + 1), eps);
          var r2 = rdp(pts.slice(index, end + 1), eps);
          return r1.slice(0, r1.length - 1).concat(r2);
        } else { return [pts[0], pts[end]]; }
      }
      return rdp(contour, epsilon);
    }

    function calculateIoU(boxA, boxB) {
      var ax1 = boxA.x, ay1 = boxA.y, ax2 = boxA.x + boxA.w, ay2 = boxA.y + boxA.h;
      var bx1 = boxB.x, by1 = boxB.y, bx2 = boxB.x + boxB.w, by2 = boxB.y + boxB.h;
      var ix1 = Math.max(ax1, bx1), iy1 = Math.max(ay1, by1);
      var ix2 = Math.min(ax2, bx2), iy2 = Math.min(ay2, by2);
      var iw = Math.max(0, ix2 - ix1), ih = Math.max(0, iy2 - iy1);
      var inter = iw * ih;
      var areaA = boxA.w * boxA.h, areaB = boxB.w * boxB.h;
      var union = areaA + areaB - inter;
      return union > 0 ? inter / union : 0.0;
    }

    // FIX: bbox extraída dos pixels reais do objeto (mat) é o GABARITO (GT) — é a posição
    // verdadeira e exata do objeto na cena. As caixas em `detBoxes` (com deslocamento
    // aleatório) representam a saída ruidosa de um detector, e são casadas ao GT mais
    // próximo por IoU.
    function measureOpenCV(mat, detBoxes) {
      var blobs = cv2_findContours(mat);
      var medidas = [];
      blobs.forEach(function(item) {
        var contour = item.contour, pixels = item.pixels;
        var area = cv2_contourArea(contour);
        if (area === 0) area = pixels.length;
        var gtBbox = cv2_boundingRect(pixels);
        var hull = cv2_convexHull(contour);
        var hull_area = cv2_contourArea(hull);
        if (hull_area === 0) hull_area = area;
        var poly = cv2_approxPolyDP(contour, 0.02);
        var solidity = hull_area > 0 ? area / hull_area : 0;
        var classe = solidity < 0.85 ? "Cruz (Côncavo)" : "Retângulo (Convexo)";
        var bestIoU = 0.0, matchedDET = { x: 0, y: 0, w: 0, h: 0 };

        detBoxes.forEach(function(d) {
          var iou = calculateIoU(gtBbox, d);
          if (iou > bestIoU) { bestIoU = iou; matchedDET = d; }
        });

        medidas.push({
          classe: classe, area: area, gtBox: gtBbox, detBox: matchedDET,
          solidity: solidity, vertices: poly.length, iou: bestIoU, ok: bestIoU >= 0.50
        });
      });
      medidas.sort(function(a, b) { return a.gtBox.x !== b.gtBox.x ? a.gtBox.x - b.gtBox.x : a.gtBox.y - b.gtBox.y; });
      medidas.forEach(function(m, i) { m.id = i + 1; });
      return medidas;
    }

    function gerarCenario() {
      matOrig = Array.from({length: H}, function(){ return Array(W).fill(0); });
      var armLen = Math.floor(Math.random() * 2) + 1, thick = 1;
      var w1 = armLen * 2 + thick, h1 = armLen * 2 + thick;
      var x1 = Math.floor(Math.random() * Math.max(1, 8 - w1)) + 1;
      var y1 = Math.floor(Math.random() * Math.max(1, H - h1)) + 1;

      for (var r = 0; r < h1; r++) {
        for (var c = 0; c < w1; c++) {
          if ((c >= armLen && c < armLen + thick) || (r >= armLen && r < armLen + thick)) { matOrig[y1 + r][x1 + c] = 180; }
        }
      }

      var w2 = Math.floor(Math.random() * 3) + 3, h2 = Math.floor(Math.random() * 3) + 3;
      var x2 = Math.floor(Math.random() * Math.max(1, W - 10 - w2)) + 10;
      var y2 = Math.floor(Math.random() * Math.max(1, H - h2)) + 1;

      for (var r = y2; r < y2 + h2; r++) {
        for (var c = x2; c < x2 + w2; c++) { matOrig[r][c] = 180; }
      }

      // Estas caixas simulam a saída de um DETECTOR real: deslocadas/imprecisas em relação
      // ao objeto verdadeiro (que será obtido depois via segmentação em matBin -> GT).
      var shiftX1 = Math.random() > 0.5 ? 1 : 0, shiftY1 = Math.random() > 0.5 ? 1 : 0;
      var shiftX2 = Math.random() > 0.6 ? -2 : 0;

      detBoxesAtuais = [
        { x: Math.max(0, x1 + shiftX1), y: Math.max(0, y1 + shiftY1), w: w1, h: h1 },
        { x: Math.max(0, x2 + shiftX2), y: y2, w: w2 + (shiftX2 !== 0 ? 2 : 0), h: h2 }
      ];

      matBin = matOrig.map(function(row) { return row.map(function(v) { return v > 50 ? 1 : 0; }); });
      medidasObjetos = measureOpenCV(matBin, detBoxesAtuais);
      renderGrid();
      renderTabela();
    }

    function inBox(r, c, box) { return r >= box.y && r < box.y + box.h && c >= box.x && c < box.x + box.w; }
    function isBoxEdge(r, c, box) { if (!inBox(r, c, box)) return false; return r === box.y || r === box.y + box.h - 1 || c === box.x || c === box.x + box.w - 1; }

    function renderGrid(){
      elGridContainer.innerHTML = '';
      var grid = document.createElement('div');
      grid.style.cssText = 'display:inline-grid;gap:1px;background:#e4dcc8;padding:1px;border-radius:6px;overflow:auto;max-width:100%;grid-template-columns:repeat(' + W + ', 22px);';
      var currentMat = stage === 0 ? matOrig : matBin;

      for (var r = 0; r < H; r++){
        for (var c = 0; c < W; c++){
          var v = currentMat[r][c], cell = document.createElement('div');
          var isGT = false, isDetOK = false, isDetFail = false;

          if (stage === 1) {
            // FIX: GT agora vem de m.gtBox (bbox real extraída dos pixels do objeto)
            if (showGT) { medidasObjetos.forEach(function(m) { if (isBoxEdge(r, c, m.gtBox)) isGT = true; }); }
            // FIX: DET agora vem de m.detBox (bbox ruidosa casada por IoU)
            if (showDET) {
              medidasObjetos.forEach(function(m) {
                if (isBoxEdge(r, c, m.detBox)) { if (m.ok) isDetOK = true; else isDetFail = true; }
              });
            }

            if (isGT && isDetOK) {
              cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:#7c3aed;color:#ffffff;border:2px double #a78bfa;box-sizing:border-box;';
            } else if (isGT && isDetFail) {
              cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:#c026d3;color:#ffffff;border:2px double #f472b6;box-sizing:border-box;';
            } else if (isDetOK) {
              cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:#059669;color:#ffffff;border:2px solid #34d399;box-sizing:border-box;';
            } else if (isDetFail) {
              cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:#dc2626;color:#ffffff;border:2px solid #f87171;box-sizing:border-box;';
            } else if (isGT) {
              cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:#2563eb;color:#ffffff;border:2px dashed #93c5fd;box-sizing:border-box;';
            } else {
              var bgBin = v === 1 ? '#26241d' : '#ffffff', fgBin = v === 1 ? '#7ee7c6' : '#8a8371';
              cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:' + bgBin + ';color:' + fgBin + ';border:none;';
            }
          } else {
            var fgCinza = v > 100 ? '#ffffff' : '#374151';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:rgb(' + (255 - v) + ',' + (255 - v) + ',' + (255 - v) + ');color:' + fgCinza + ';border:none;';
          }
          cell.textContent = v;
          grid.appendChild(cell);
        }
      }
      elGridContainer.appendChild(grid);
    }

    function renderTabela(){
      elTbody.innerHTML = '';
      if (medidasObjetos.length === 0) {
        elTbody.innerHTML = '<tr><td colspan="9" style="padding:12px;color:#8a8371;text-align:center;">Nenhum objeto detectado na cena.</td></tr>';
        return;
      }
      medidasObjetos.forEach(function(m, i){
        var tr = document.createElement('tr');
        if (i % 2 === 1) tr.style.background = '#fafaf7';
        // FIX: bboxDet vem de m.detBox (caixa ruidosa) e bboxGT vem de m.gtBox (caixa real)
        var bboxDet = '(' + m.detBox.x + ',' + m.detBox.y + ',' + m.detBox.w + ',' + m.detBox.h + ')';
        var bboxGT = '(' + m.gtBox.x + ',' + m.gtBox.y + ',' + m.gtBox.w + ',' + m.gtBox.h + ')';
        var statusHtml = m.ok ? '<span style="color:#27ae60;font-weight:bold;">✔ True</span>' : '<span style="color:#e74c3c;font-weight:bold;">✖ False</span>';

        tr.innerHTML = 
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;"><b>' + m.id + '</b></td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;"><b>' + m.classe + '</b></td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.area.toFixed(1) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.solidity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.vertices + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + bboxDet + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + bboxGT + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.iou.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + statusHtml + '</td>';
        elTbody.appendChild(tr);
      });
    }

    elToggleGT.addEventListener('click', function(){
      showGT = !showGT;
      if (showGT) {
        this.style.background = '#2563eb'; this.style.borderColor = '#1d4ed8'; this.style.color = '#ffffff';
        this.querySelector('span').textContent = '🟦';
      } else {
        this.style.background = '#f1ead7'; this.style.borderColor = '#d4cebe'; this.style.color = '#5e5a4a';
        this.querySelector('span').textContent = '⬜';
      }
      renderGrid();
    });

    elToggleDET.addEventListener('click', function(){
      showDET = !showDET;
      if (showDET) {
        this.style.background = '#059669'; this.style.borderColor = '#047857'; this.style.color = '#ffffff';
        this.querySelector('span').textContent = '🟩';
      } else {
        this.style.background = '#f1ead7'; this.style.borderColor = '#d4cebe'; this.style.color = '#5e5a4a';
        this.querySelector('span').textContent = '⬜';
      }
      renderGrid();
    });

    elBtnRand.addEventListener('click', gerarCenario);

    function setStage(s, btn){
      [root.querySelector('#ep0811_stage0'), root.querySelector('#ep0811_stage1')].forEach(function(b){
        b.style.background = 'transparent'; b.style.color = '#8a8371'; b.style.fontWeight = '600';
      });
      btn.style.background = '#26241d'; btn.style.color = '#fbf7ee'; btn.style.fontWeight = '700';
      stage = s; renderGrid();
    }

    root.querySelector('#ep0811_stage0').addEventListener('click', function(){ setStage(0, this); });
    root.querySelector('#ep0811_stage1').addEventListener('click', function(){ setStage(1, this); });

    gerarCenario();
  }

  function tryInit(){
    var root = document.getElementById('sim-ep0811');
    if (root) initSim(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figure 8.11:** Simulateur EP08_11 : Classification géométrique avec contrôles indépendants des *overlays BBox* (GT et DET)


In [ ]:
%%writefile EP08_11.py
# Code Python

In [ ]:
TestSuite("EP08_11.py").run()

### EP08_12 🔴 Segmentation d'instances sur image réelle : objets géométriques

L'exemple de segmentation classique de ce chapitre a séparé des « instances » de pièces de monnaie par déconnexion spatiale dans le masque binaire résultant du seuillage d'Otsu. Dans cet exercice, vous appliquerez la même idée — mais cette fois sur une image réelle avec des objets géométriques variés — en enchaînant le prétraitement, la binarisation, l'extraction de contours (`cv2.findContours`) et la validation du résultat par rapport à une référence de *bounding boxes*.

Contrairement à l'exercice précédent (étiquetage sur un masque déjà prêt), ici vous partez de l'**image originale** : la qualité de votre segmentation dépend directement des choix de prétraitement (filtrage, seuillage, opérations morphologiques) effectués avant d'étiqueter les composants.

#### 📋 Directives d'implémentation

1. **Entrée :** utiliser l'image `00000.jpg`.
2. **Prétraitement et segmentation :** appliquer les étapes nécessaires (filtrage, binarisation et opérations morphologiques) pour séparer automatiquement les objets du fond, sans recadrages manuels.
3. **Étiquetage et mesure :** pour chaque objet segmenté, déterminer :
   - l'aire ;
   - le centre de masse (centroïde) ;
   - le type, selon l'ensemble `obj2`.
4. **Annotation visuelle :** écrire, à l'intérieur de chaque objet, son aire et le sigle du type (`obj2`).
5. **Validation (IoU) :** calculer l'*Intersection over Union* (IoU) entre la *bounding box* détectée (`cv2.boundingRect`) et la *bounding box* de référence du type correspondant. Un objet est considéré comme correctement segmenté uniquement s'il existe exactement une *bounding box* du type correct avec **IoU ≥ 0,5**.
6. **Sortie :** imprimer, pour chaque objet détecté, son identifiant, son type et s'il a été validé avec succès (`acertou=1`) ou non. L'impression doit suivre l'ordre des classes de `obj2` (0=Tria … 8=Cruz) ; au sein de la même classe, trier les objets par coordonnée verticale du centroïde (`cy`) croissante. À la fin, imprimer la précision globale.

#### 📌 Contraintes informatiques

* **Sans recadrage manuel :** toute la segmentation doit être effectuée sur l'image complète.
* **Ensemble de classes fixe :**
  ```python
  obj  = ['Triangulo','Quadrado','Pendagono','Hexagono','Heptagono','Circulo',
          'Elipse','Estrela','Cruz']
  obj2 = ['Tria','Quad','Pent','Hexa','Hept','Circ','Elip','Estr','Cruz']
  ```
* **Dimensions de l'image :** 608×608 pixels — utilisées pour dénormaliser les coordonnées du fichier TXT.
* **Validation par centre de masse :** un objet n'est considéré comme correctement segmenté que si son centroïde se trouve strictement à l'intérieur de la *bounding box* de référence correspondant au même type d'objet.

#### 🧠 Fondements théoriques

| Élément | Rôle dans la segmentation d'instances |
|---|---|
| Prétraitement (filtrage, seuillage) | Étape qui produit le masque binaire à partir de l'image d'intensité originale |
| `cv2.findContours` | Extrait les contours des composants connectés dans le masque binaire |
| Moments géométriques (`cv2.moments`) | Permettent de calculer le centre de masse (centroïde) de chaque contour |
| `approxPolyDP` / sommets | Aide à la classification du type d'objet (nombre de côtés approximatif) |
| Validation via *bounding box* | Confirme si l'instance segmentée correspond spatialement à un objet de la référence, en mesurant la précision de la méthode |

#### 📌 Exemple de sortie attendue

```
Objeto 1: tipo=Tria, validado=True
...
Acurácia: 88.89%
```

**Paramètres fixes pour la reproductibilité :** pour que la sortie corresponde à la référence de correction automatique, utilisez exactement : filtre d'aire minimale de 300 pixels ; `cv2.approxPolyDP` avec `epsilon = 0.02 * périmètre` ; seuil de solidité 0,92 et nombre de sommets ≥ 9 (avec ≥ 11 pour différencier Cruz de Estrela) pour les formes concaves ; rapport d'aspect 1,15 pour différencier Círculo de Elipse ; seuil IoU = 0,5 lors de la validation.

#### 📌 Fichiers de référence (`.jpg` et `.txt`)

Pour le débogage local, deux fichiers de référence sont fournis (inclus dans cette livraison ; lors de leur intégration au dépôt du chapitre, enregistrez-les dans `all/cap08/dados/EP08/`) :

* 📥 **Image (`00000.jpg`)** : image d'objets géométriques utilisée comme entrée de l'exercice. L'objectif est de segmenter automatiquement chaque objet, de déterminer son type et de calculer ses mesures.
* 📥 **Référence (`00000.txt`)** : fichier contenant les *bounding boxes* normalisées des objets présents dans l'image. Chaque ligne contient l'identifiant de la classe et les coordonnées normalisées des coins supérieur gauche et inférieur droit, utilisé pour valider automatiquement la segmentation.

La [Figure 8.12](#fig-08-ep12) présente l'image d'entrée et la même image avec les *bounding boxes* dessinées à partir du fichier de référence.

In [ ]:
import os
import urllib.request
from morph import mm

def garantir_e_baixar(nome):
    pasta = "dados/EP12"
    caminho = os.path.join(pasta, nome)

    os.makedirs(pasta, exist_ok=True)

    if not os.path.exists(caminho):
        url = (
            "https://raw.githubusercontent.com/"
            "fzampirolli/pdi-vc/master/all/cap08/dados/EP12/"
            + nome
        )
        print(f"Téléchargement {nome}...")
        urllib.request.urlretrieve(url, caminho)

    return caminho

img_arq = garantir_e_baixar("00000.jpg")
txt_arq = garantir_e_baixar("00000.txt")

img = mm.read(img_arq)
img_bb = mm.showBoundBox(img, txt_arq, fmt="yolo", show=False)

mm.show(
    [img, img_bb],
    titles=[
        "Image originale",
        "Bounding boxes de la correction"
    ],
    cols=2,
    figsize=(10,5)
)

**Figure 8.12:** Simulateur EP08_12 : Image utilisée dans l


In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0812" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧮 Simulateur EP08_12 : Précision de segmentation sur objets multiples</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">🟢 réussi si IoU ≥ seuil et type correct</span>
  </div>


  <div style="padding:20px;background:white;overflow:auto">

    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">
      Chaque forme a une <i>boundbox</i> de référence (rectangle en pointillés, juste autour de la forme) et une <i>boundbox</i> détectée (rectangle plein, décalée/bruitée). Ajustez le bruit, le biais et le seuil d'IoU pour voir la validation changer.
    </p>

    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:18px;display:grid;grid-template-columns:1fr 1fr;gap:16px;">
      <div>
        <div style="display:flex;justify-content:space-between;margin-bottom:6px;"><label style="font-size:12px;font-weight:bold;color:#c0392b;">bruit de segmentation (px, jitter max par côté)</label><span id="ep0812_ruido_v" style="font-family:monospace;font-weight:bold;color:#c0392b;">0</span></div>
        <input id="ep0812_ruido" style="width:100%;accent-color:#c0392b;" max="20" min="0" step="1" type="range" value="0">
      </div>
      <div>
        <div style="display:flex;justify-content:space-between;margin-bottom:6px;"><label style="font-size:12px;font-weight:bold;color:#2980b9;">biais systématique en x (px)</label><span id="ep0812_bias_v" style="font-family:monospace;font-weight:bold;color:#2980b9;">0</span></div>
        <input id="ep0812_bias" style="width:100%;accent-color:#2980b9;" max="20" min="-20" step="1" type="range" value="0">
      </div>
      <div>
        <div style="display:flex;justify-content:space-between;margin-bottom:6px;"><label style="font-size:12px;font-weight:bold;color:#27ae60;">seuil d'IoU</label><span id="ep0812_thr_v" style="font-family:monospace;font-weight:bold;color:#27ae60;">0.50</span></div>
        <input id="ep0812_thr" style="width:100%;accent-color:#27ae60;" max="0.9" min="0.1" step="0.05" type="range" value="0.5">
      </div>
      <div style="display:flex;align-items:center;gap:8px;">
        <input id="ep0812_erro" type="checkbox" style="accent-color:#8e44ad;width:16px;height:16px;">
        <label style="font-size:12px;font-weight:bold;color:#8e44ad;">simuler une erreur de classification (2 objets avec type échangé)</label>
      </div>
    </div>

<div id="ep0812_svg" style="width:100%;max-width:420px;margin:0 auto 16px auto;"></div>

    <table style="width:100%;border-collapse:collapse;font-size:11px;font-family:monospace;margin-bottom:12px;">
      <thead>
        <tr style="background:#f3efe6;">
          <th style="padding:4px;border:1px solid #ddd;">id</th>
          <th style="padding:4px;border:1px solid #ddd;">type réel</th>
          <th style="padding:4px;border:1px solid #ddd;">type détecté</th>
          <th style="padding:4px;border:1px solid #ddd;">IoU</th>
          <th style="padding:4px;border:1px solid #ddd;">≥ seuil</th>
          <th style="padding:4px;border:1px solid #ddd;">réussi</th>
        </tr>
      </thead>
      <tbody id="ep0812_tbody"></tbody>
    </table>

    <div id="ep0812_debug" style="background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:12px;color:#1565c0;text-align:center;"></div>
  </div>
</div>
<script>

function svgNS(tag){
  var SVG_NS = "http" + "://www.w3.org/2000/svg";
  return document.createElementNS(SVG_NS, tag);
}

(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var ruidoEl = root.querySelector('#ep0812_ruido'), ruidovEl = root.querySelector('#ep0812_ruido_v');
    var biasEl = root.querySelector('#ep0812_bias'), biasvEl = root.querySelector('#ep0812_bias_v');
    var thrEl = root.querySelector('#ep0812_thr'), thrvEl = root.querySelector('#ep0812_thr_v');
    var erroEl = root.querySelector('#ep0812_erro');
    var svgWrap = root.querySelector('#ep0812_svg');
    var svgEl = svgNS('svg');
    svgEl.setAttribute('viewBox', '0 0 360 340');
    svgEl.setAttribute('style', 'width:100%;display:block;background:#0d0d0d;border-radius:12px;border:1px solid #333;');
    svgWrap.appendChild(svgEl);

    var tbody = root.querySelector('#ep0812_tbody');
    var dbg = root.querySelector('#ep0812_debug');

    var TIPOS = ['Circ','Tria','Quad','Cruz','Pent','Hexa','Hept','Estr','Elip'];
    var CORES = ['#a33a5b','#8fe3b0','#c98a7a','#5c4a5e','#3fbf5f','#d9c832','#e08a2b','#5c5470','#4a5a3a'];
    var FATOR = [1.0, 1.3, 0.7, 1.5, 0.9, 1.1, 0.6, 1.4, 0.8]; // sensibilidade individual ao ruído (fixa)
    var ANGS  = [30, 160, 260, 5, 200, 90, 340, 130, 240];      // direção fixa do erro por objeto (graus)
    var SCALE = [0.9, 1.15, 0.8, 1.2, 1.0, 0.95, 1.1, 1.25, 0.85]; // fator de encolhimento/expansão do bbox (fixo)

    var OBJS = [
      {cx:55,  cy:45,  r:22},
      {cx:150, cy:70,  r:24},
      {cx:250, cy:50,  r:22},
      {cx:320, cy:130, r:20},
      {cx:90,  cy:190, r:24},
      {cx:190, cy:220, r:24},
      {cx:275, cy:190, r:24},
      {cx:315, cy:270, r:22},
      {cx:150, cy:150, r:18}
    ];

    function svgShape(tipo, cx, cy, r, cor){
      var s = '';
      if(tipo==='Circ'){
        s = '<circle cx="'+cx+'" cy="'+cy+'" r="'+r+'" fill="'+cor+'"/>';
      } else if(tipo==='Elip'){
        s = '<ellipse cx="'+cx+'" cy="'+cy+'" rx="'+(r*1.1)+'" ry="'+(r*0.65)+'" fill="'+cor+'"/>';
      } else if(tipo==='Quad'){
        s = '<rect x="'+(cx-r*0.8)+'" y="'+(cy-r*0.8)+'" width="'+(r*1.6)+'" height="'+(r*1.6)+'" fill="'+cor+'"/>';
      } else if(tipo==='Cruz'){
        var w = r*0.5, l = r*1.4;
        s = '<g fill="'+cor+'">'+
            '<rect x="'+(cx-w/2)+'" y="'+(cy-l/2)+'" width="'+w+'" height="'+l+'"/>'+
            '<rect x="'+(cx-l/2)+'" y="'+(cy-w/2)+'" width="'+l+'" height="'+w+'"/></g>';
      } else {
        var sides = {Tria:3, Pent:5, Hexa:6, Hept:7}[tipo];
        var isStar = (tipo==='Estr');
        var pts = [];
        if(isStar){
          var spikes=5, outer=r, inner=r*0.45;
          for(var i=0;i<spikes*2;i++){
            var rad = (i%2===0)?outer:inner;
            var ang = Math.PI/spikes*i - Math.PI/2;
            pts.push((cx+rad*Math.cos(ang)).toFixed(1)+','+(cy+rad*Math.sin(ang)).toFixed(1));
          }
        } else {
          for(var i=0;i<sides;i++){
            var ang = 2*Math.PI/sides*i - Math.PI/2;
            pts.push((cx+r*Math.cos(ang)).toFixed(1)+','+(cy+r*Math.sin(ang)).toFixed(1));
          }
        }
        s = '<polygon points="'+pts.join(' ')+'" fill="'+cor+'"/>';
      }
      return s;
    }

    // bbox "verdadeiro": retângulo justo em torno da forma (sem folga artificial)
    function trueBBox(tipo, cx, cy, r){
      var hw = r, hh = r;
      if(tipo==='Elip'){ hw = r*1.1; hh = r*0.65; }
      else if(tipo==='Quad'){ hw = r*0.8; hh = r*0.8; }
      else if(tipo==='Cruz'){ hw = r*0.7; hh = r*0.7; }
      return [cx-hw, cy-hh, cx+hw, cy+hh];
    }

    function iou(a, b){
      var x1 = Math.max(a[0], b[0]), y1 = Math.max(a[1], b[1]);
      var x2 = Math.min(a[2], b[2]), y2 = Math.min(a[3], b[3]);
      var inter = Math.max(0, x2-x1) * Math.max(0, y2-y1);
      var areaA = (a[2]-a[0])*(a[3]-a[1]);
      var areaB = (b[2]-b[0])*(b[3]-b[1]);
      var uni = areaA + areaB - inter;
      return uni > 0 ? inter/uni : 0;
    }

    function render(){
      var ruido = parseInt(ruidoEl.value);
      var bias = parseInt(biasEl.value);
      var thr = parseFloat(thrEl.value);
      var erroAtivo = erroEl.checked;
      ruidovEl.textContent = ruido;
      biasvEl.textContent = bias;
      thrvEl.textContent = thr.toFixed(2);

      var svgContent = '';
      var rows = '';
      var acertos = 0;

      OBJS.forEach(function(o, i){
        var tipoReal = TIPOS[i];
        var cor = CORES[i];

        var gtBox = trueBBox(tipoReal, o.cx, o.cy, o.r);
        svgContent += '<g opacity="0.9">'+svgShape(tipoReal, o.cx, o.cy, o.r, cor)+'</g>';
        svgContent += '<rect x="'+gtBox[0]+'" y="'+gtBox[1]+'" width="'+(gtBox[2]-gtBox[0])+'" height="'+(gtBox[3]-gtBox[1])+'" fill="none" stroke="#aaa" stroke-dasharray="4,3" stroke-width="1.2"/>';

        // bbox detectado: escala fixa individual + jitter (ruído) + viés em x
        var scl = SCALE[i];
        var mag = ruido * FATOR[i];
        var ang = ANGS[i] * Math.PI/180;
        var jx = mag*Math.cos(ang), jy = mag*Math.sin(ang);
        var dw = (gtBox[2]-gtBox[0]) * scl, dh = (gtBox[3]-gtBox[1]) * scl;
        var dcx = o.cx + jx + bias, dcy = o.cy + jy;
        var detBox = [dcx-dw/2, dcy-dh/2, dcx+dw/2, dcy+dh/2];

        var val = iou(gtBox, detBox);
        var passaLimiar = val >= thr;

        var corDet = passaLimiar ? '#27ae60' : '#c0392b';
        svgContent += '<rect x="'+detBox[0]+'" y="'+detBox[1]+'" width="'+(detBox[2]-detBox[0])+'" height="'+(detBox[3]-detBox[1])+'" fill="none" stroke="'+corDet+'" stroke-width="1.6"/>';

        var tipoDetectado = tipoReal;
        if(erroAtivo && (i===1 || i===6)){
          tipoDetectado = TIPOS[(i+2)%TIPOS.length];
        }
        var tipoCorreto = (tipoDetectado === tipoReal);
        var acertou = passaLimiar && tipoCorreto;
        if(acertou) acertos++;

        svgContent += '<text x="'+(o.cx)+'" y="'+(gtBox[1]-6)+'" font-size="9" fill="#ccc" text-anchor="middle" font-family="monospace">'+ (i+1) +'</text>';

        rows += '<tr>'+
          '<td style="padding:4px;border:1px solid #ddd;text-align:center;">'+(i+1)+'</td>'+
          '<td style="padding:4px;border:1px solid #ddd;text-align:center;">'+tipoReal+'</td>'+
          '<td style="padding:4px;border:1px solid #ddd;text-align:center;'+(tipoCorreto?'':'color:#c0392b;font-weight:bold;')+'">'+tipoDetectado+'</td>'+
          '<td style="padding:4px;border:1px solid #ddd;text-align:center;">'+val.toFixed(2)+'</td>'+
          '<td style="padding:4px;border:1px solid #ddd;text-align:center;">'+(passaLimiar?'sim':'não')+'</td>'+
          '<td style="padding:4px;border:1px solid #ddd;text-align:center;'+(acertou?'color:#27ae60;font-weight:bold;':'color:#c0392b;font-weight:bold;')+'">'+(acertou?'✔':'✘')+'</td>'+
          '</tr>';
      });

      svgEl.innerHTML = svgContent;
      tbody.innerHTML = rows;

      var acc = (acertos/OBJS.length*100).toFixed(1);
      dbg.textContent = 'Objetos validados: '+acertos+' / '+OBJS.length+'  →  Acurácia = '+acc+'%';
    }

    ruidoEl.addEventListener('input', render);
    biasEl.addEventListener('input', render);
    thrEl.addEventListener('input', render);
    erroEl.addEventListener('change', render);
    render();
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0812');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figure 8.13:** Simulateur EP08_12 : Précision de Segmentation sur Objets Multiples (IoU)


In [ ]:
%%writefile EP08_12.py
# Code Python

In [ ]:
TestSuite("EP08_12.py").run()